# Đánh giá chung: mô hình đề xuất và TFCDiff

Chạy trên Kaggle với **GPU**. Add ba file làm Kaggle Dataset: `common_qtdb_mixed_bwem_ma_3600_ntype1.npz`, `best_qtdb_mixed_bwem_ma_3600_fresh.pth` và `comparative_checkpoints_bundle.zip`. ZIP chứa TFCDiff và các mô hình so sánh; checkpoint Proposed mới phải tải lên riêng. Notebook dùng cùng 5.012 mẫu QTDB, cùng tín hiệu sạch/nhiễu và xuất dự đoán, metrics, biểu đồ. Hai mô hình giữ đúng quy trình suy luận của từng nghiên cứu: proposed = EMA + DDIM 50 bước + chuẩn hóa maxabs theo tín hiệu nhiễu; TFCDiff = DCT ortho → 1000 hệ số / 3 → DDPM 50 bước → IDCT. Nguồn mô hình được đóng gói trong notebook và kiểm tra SHA256.

**Lưu ý công bằng:** hai sampler khác nhau vì đó là cấu hình suy luận thực tế của mỗi checkpoint. Metrics được tính lại trên cùng các tín hiệu và cùng mã chấm điểm. Để tái lập, dùng cùng phiên bản PyTorch/CUDA và seed; dự đoán ngẫu nhiên có thể thay đổi nhẹ giữa môi trường.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, sys, time, types, zlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_NAME = 'common_qtdb_mixed_bwem_ma_3600_ntype1.npz'
PROPOSED_NAME = 'best_qtdb_mixed_bwem_ma_3600_fresh.pth'
TFCDIFF_NAME = 'model.pth'
# Điền đường dẫn tuyệt đối nếu Kaggle Input có nhiều file trùng tên.
DATA_PATH = None
PROPOSED_PATH = None
TFCDIFF_PATH = None
SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
OUT = Path('/kaggle/working/tfcdiff_mixed_evaluation') if Path('/kaggle/working').exists() else Path.cwd() / 'tfcdiff_mixed_evaluation'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 2024
PROPOSED_BATCH_SIZE = 4
TFCDIFF_BATCH_SIZE = 4
METRIC_BATCH_SIZE = 50  # batch cố định cho PRD theo implementation trong utils.py
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '| Torch:', torch.__version__, '| Output:', OUT)
if DEVICE.type != 'cuda':
    raise RuntimeError('Cần bật GPU trong Kaggle Notebook Settings để chạy đủ 5.012 mẫu.')

def locate(name, explicit=None):
    if explicit:
        p = Path(explicit)
        if not p.is_file(): raise FileNotFoundError(p)
        return p
    found = sorted({p.resolve() for root in SEARCH_ROOTS if root.exists() for p in root.rglob(name) if p.is_file()})
    if len(found) != 1:
        raise RuntimeError(f'{name}: tìm thấy {len(found)} file: {found}. Hãy đặt đường dẫn *_PATH ở cell này.')
    return found[0]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

data_path = locate(DATA_NAME, DATA_PATH)
proposed_path = locate(PROPOSED_NAME, PROPOSED_PATH)
# TFCDiff checkpoint phải đúng fingerprint; DeScoD cũng có file tên model.pth.
EXPECTED_TFCDIFF_SHA256 = '11ce74ff739e6984e2c42aa1919662c4fd66c43e910e11a59bc3117c1d24509f'
try:
    tfcdiff_path = locate(TFCDIFF_NAME, TFCDIFF_PATH)
except RuntimeError:
    if TFCDIFF_PATH is not None:
        raise
    tfcdiff_path = None
if tfcdiff_path is None:
    import zipfile
    try:
        baseline_bundle = locate('comparative_checkpoints_bundle.zip')
    except RuntimeError as exc:
        raise RuntimeError('Thiếu TFCDiff checkpoint: hãy add comparative_checkpoints_bundle.zip bản mới hoặc check_points/model.pth vào Kaggle Input.') from exc
    with zipfile.ZipFile(baseline_bundle) as archive:
        try:
            baseline_bytes = archive.read('TFCDiff/model.pth')
        except KeyError as exc:
            raise RuntimeError('ZIP cũ chưa chứa TFCDiff/model.pth. Hãy upload lại comparative_checkpoints_bundle.zip bản mới.') from exc
    if hashlib.sha256(baseline_bytes).hexdigest() != EXPECTED_TFCDIFF_SHA256:
        raise RuntimeError('TFCDiff checkpoint trong ZIP không đúng SHA256.')
    tfcdiff_path = OUT / 'tfcdiff_model.pth'
    tfcdiff_path.write_bytes(baseline_bytes)
elif sha256_file(tfcdiff_path) != EXPECTED_TFCDIFF_SHA256:
    raise RuntimeError(f'model.pth tìm thấy là checkpoint khác TFCDiff: {tfcdiff_path}. Hãy add ZIP mới hoặc đặt TFCDIFF_PATH đúng.')
print('Data:', data_path, '\nProposed:', proposed_path, '\nTFCDiff:', tfcdiff_path)


In [ ]:
import importlib.util, gc, shutil, zipfile

COMPARATIVE_BUNDLE_NAME = 'comparative_checkpoints_bundle.zip'
COMPARATIVE_BUNDLE_PATH = None  # có thể đặt đường dẫn tuyệt đối
WEIGHT_HASHES = {'DeScoD/model.pth': '83e9c9d6747cca354c887c298e9ca19845eb91dd2eec165abf381ab975f58c5d', 'TFCDiff/model.pth': '11ce74ff739e6984e2c42aa1919662c4fd66c43e910e11a59bc3117c1d24509f', 'DeepFilter-master/DRNN_.best.weights.h5': 'b3f258ad6eaef1520776d86471bc585b3f905d691edfd6ee7b63ba4efeb261e6', 'DeepFilter-master/FCN_DAE_.best.weights.h5': 'ce130db38321192b76d0f081db3d3e70242d610831d1233246ee8655fd9a5268', 'DeepFilter-master/Multibranch_LANL_.best.weights.h5': '949219604fc469451e64844445318b274b653c7b990e32fe945cf186b01156e7', 'DeepFilter-master/Multibranch_LANLD_.best.weights.h5': '3f9207b5d0681182bef06d9904a3f75076e98a952c0b32b1f846175137ab5cfa', 'DeepFilter-master/Vanilla_L_.best.weights.h5': '06c8bcfdf05862fa66b1acfb213792a6df6dc826026fbcbee8ea3360893582d4', 'DeepFilter-master/Vanilla_NL_.best.weights.h5': '5f7fb365a47457c5e0a68930b6ed0fb07f8272c2641166cb08f1398d20a122b4', 'TCDAE/Transformer_DAE_.best.weights.h5': '7bc3981050887fe277f0f25d2a4e4d11b980765c1b9eb34ddfa95f0202e0d326', 'CBAM-DAE/Attention_Skip_DAE_.best.weights.h5': '59a7c179a2908caeb11e6654162ed9634b5144d653be8daf6918500bc2795662', 'CBAM-DAE/ECA_Skip_DAE_.best.weights.h5': 'e519fc4098fb8ac611b2fb83699189b11916dd6c5a33415f25e9e8fe3c23cc37', 'CBAM-DAE/Transformer_DAE_.best.weights.h5': '6989b3f0fd02376b96783aca24f82fd51c073e4880ce96cdebce2850ecdbaa2c'}
WEIGHT_ROOT = OUT / 'comparative_weights'
WEIGHT_ROOT.mkdir(parents=True, exist_ok=True)
REQUIRED_COMPARATIVE = {k:v for k,v in WEIGHT_HASHES.items() if not k.startswith('TFCDiff/')}
try:
    comparison_bundle_path = locate(COMPARATIVE_BUNDLE_NAME, COMPARATIVE_BUNDLE_PATH)
except RuntimeError:
    if COMPARATIVE_BUNDLE_PATH is not None:
        raise
    comparison_bundle_path = None
comparison_bundle_hash = sha256_file(comparison_bundle_path) if comparison_bundle_path else None
if comparison_bundle_path:
    with zipfile.ZipFile(comparison_bundle_path) as archive:
        manifest = json.loads(archive.read('sha256.json'))
        for rel, expected in REQUIRED_COMPARATIVE.items():
            if manifest.get(rel) != expected or rel not in archive.namelist():
                raise RuntimeError(f'ZIP thiếu hoặc sai checkpoint {rel}. Hãy upload đúng comparative_checkpoints_bundle.zip.')
            raw = archive.read(rel)
            if hashlib.sha256(raw).hexdigest() != expected:
                raise RuntimeError(f'ZIP hỏng checkpoint {rel}.')
            path = WEIGHT_ROOT / rel
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(raw)
else:
    # Hỗ trợ các file đã giải nén, kể cả tên DeScoD người dùng đổi thành des-cod-model.pth.
    valid_names = {Path(rel).name for rel in REQUIRED_COMPARATIVE} | {'des-cod-model.pth'}
    candidates = sorted({p.resolve() for root in SEARCH_ROOTS if root.exists()
                         for p in root.rglob('*') if p.is_file() and p.name in valid_names})
    by_hash = {}
    for candidate in candidates:
        by_hash.setdefault(sha256_file(candidate), []).append(candidate)
    missing = []
    for rel, expected in REQUIRED_COMPARATIVE.items():
        matches = by_hash.get(expected, [])
        if not matches:
            missing.append(rel)
            continue
        target = WEIGHT_ROOT / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        if matches[0] != target.resolve():
            shutil.copy2(matches[0], target)
    if missing:
        raise RuntimeError('Thiếu Comparative_study checkpoints trước khi chạy suy luận: ' + ', '.join(missing) + '. Hãy add ZIP hoặc các H5/PTH vào Kaggle Input.')
print('Preflight OK:', len(REQUIRED_COMPARATIVE), 'Comparative_study checkpoints;',
      'ZIP' if comparison_bundle_path else 'loose files')
if importlib.util.find_spec('tensorflow') is None or importlib.util.find_spec('keras') is None:
    raise RuntimeError('Kaggle image thiếu TensorFlow/Keras để chạy H5. Cài thư viện trước khi chạy suy luận dài.')


In [ ]:
# Đọc metadata và kiểm tra đúng bundle đã dùng khi train.
data_hash = sha256_file(data_path)
proposed_hash = sha256_file(proposed_path)
tfcdiff_hash = sha256_file(tfcdiff_path)
checkpoint = torch.load(proposed_path, map_location='cpu', weights_only=True)
assert data_hash == checkpoint['data_fingerprints']['bundle_file'], 'NPZ khác với dữ liệu của checkpoint!'
assert checkpoint['config']['experiment']['n_type'] == 1
assert checkpoint['config']['experiment']['input_length'] == 3600
assert checkpoint['config']['diffusion']['num_steps'] == 50
with np.load(data_path, allow_pickle=False) as bundle:
    clean = np.ascontiguousarray(bundle['clean_test'], dtype=np.float32)
    noisy = np.ascontiguousarray(bundle['noisy_test'], dtype=np.float32)
    sample_id = bundle['sample_id_test'].astype(np.int64)
    record_id = bundle['record_id_test'].astype(str)
    q_test = bundle['q_test'].astype(np.float64)
    if 'record_id_train' in bundle.files and 'record_id_val' in bundle.files:
        train_records = set(bundle['record_id_train'].astype(str))
        val_records = set(bundle['record_id_val'].astype(str))
        assert not set(record_id).intersection(train_records | val_records), 'Leakage giữa test và train/val'
assert clean.shape == noisy.shape == (5012, 1, 3600)
assert len(np.unique(sample_id)) == len(sample_id)
assert np.isfinite(clean).all() and np.isfinite(noisy).all()
assert len(record_id) == len(q_test) == len(sample_id)
print('Verified:', len(sample_id), 'test samples; data SHA256:', data_hash)
print('Checkpoint epoch:', checkpoint['epoch'], 'best validation loss:', checkpoint['best_validation_loss'])


In [ ]:
# Khôi phục nguyên byte mã nguồn mô hình, không cần git clone hay pip install.
PAYLOAD = {'tfcdiff/unet.py': 'eNrtHNtu3Mb13YD/gbFQiCtTu9qVkwejLODEVlHYDoI6RR4EgaDI0S4h3kwOpVX+regn9Rd6zlzIM8PZmyPXSWtBFy45c+63OTNUVtRVw70i5qunTzL5gVdNAp9umqqQ1556UJbmmGlZTm+6MuFZVca5F7fehZqWlW3NEq4nZq0e1gMou6J+wCllrVFlRTHN4wfWtF6PJVrwrs6ZiTdKATRMhT9Pnzx9krIbj62zlrf+evLy6RMPvhrGu6b01oDaKyvu/ViVTI+Fn7jLuX8X54GX6hlZDwTu65sEFNw1QKf+BOcMrPnpxGN5y7wUESV53LbeD1V5N099kJO80nCRjCjKyoxHkd+y/CbwTuJm2cKfk9t7vKIUtF3NGn8y7WfYY8lQgDVtWMt4VMdNXDAO4vTVgAH3aATOoyiBYsQ1vY2zIiuXUVk1RZxLWqf3LFuuOMEKchAPrjPQCpE4AUiB/sqaqlWwcMpkENi7rGRxgwKTV19OYL9rYR15vzCPNxlLwSOAwDqPE6aszbvP+Mp7/VDGRZZEibgXeNcduCH30oq15TH3QD43QCV60T3Lhe+qp+C74H7xEuyYXedd0wBHHo/b21brKFWgY85ZKSwfaH1fpV3OtqgrzYrAa2IYD4p4G3icFUBEDL7EAk+MlJIKf246NlbmCGkgpEh0TEQMVDIMFgOKP52H4dzSeny3rKsq90KU9as0rnl2x17dLX+Cm+C0c1NnQP834bmlpFWWpqyM4BlAyUruw9WJZHLynODDuLBt6luLtJtkDndV9BCSG4YH3hzUCZYQXsQAllB5pAyrlCx9H/Nk9SNoeZH6ajrYSclG5n+TLAZsFNHbAZfQijWPyBfmk0+G4IhuLREIIEJ9WZxnvzI1ynI/1wDbA8GWPcw6EmQhbBHAWPhEuIbUxOMyYX7hBaaQ7OHUC8GNcBqP/EJ5NIhmst94lF/gnRlMdXUacxYRmY140nGCjPkmnLtESPVwGnrn5pCjukHLPP5hFZfg1XQwr14egx/xxrfhTAS1A70gYIicqfLlNaVzDbqn/gRJ2PEQLHr04AICb95tGr+AB9O7jN3762kLyvfPJoF3SoWuMvHFtK1ueBGv/fXM5kNqacgvZljcP3DBr6jqQOm3rClZHiFBKpyFZ9PFt0KKWcpC8Jc6TlMImniZZjkOKfF62VRd3YbUocDDwhdGKAzPX+wTDe3ovjMUAv1/UgSEZ5YbyxAEv8f3kWf5DK+s50QUMIZ8ssZJyaBixYX1VEkLHqurERVShIIMeWmNkHzBc3lhPRVpNhQyt568dQTePr0gulGeG3KYncJ65+4hSf3JSPyTrh18WTc3cZmW/ltiVnAxm0n6DRsDk2/Yxy6DGiRaNnFqx2EIEsiZKyooxh3oRUlB0E8mWzMVhSbr6C8Q3TOM7g1GMBmr3k421Eu6COvKDCscowq7zK4OjcKmVUwdMyZ7xkkVpCJqYiZ0IxJeY2JScSaB0A0WEXg5K5dQ3IWeColW5FzLeDnHOKkHBx4Z1BslEYqcI4WqAiwJH8tlw5bIcj9V2lBR+COGAgVfhe2BgxMaT4I+usxmxH+DUVAJDi+aB3KVvW4jdiipJb2ndkIHYmsR/y6mKl+sNYuhLRgV1U0C+qxA4uCQH2j4C8alhPnVZxIjKPZZhQjyZJD7dr/+BPZQ5F+EKe/EM9gaIPdcyAvb9ALPtD01SngPKHxcTcgBesF85P1UtZlscLwpk0qkqg9V1yTs3//6J3x7K87r9iUEb1h1ddcgyWKWFyU/xYv7+I5h1J5d59X1rIhbiMGztkmGB1CqsnxaP+j6ZIxt3xpl6/p3sjnlb4pdZZW1LMrZHTO6IeR2SK4dHpRUXdmHGkQ4m3kLQgdnde+gsYzsYgpwwx9qZkAXd3aYU8russSaJm5NvJkkhniDVmVIGZp2ZfuxYwxMw0Q1n4D9SUrZuvZPsV02zaulP2cv8BHyQiafGQl1QCUhJDG3wF/KB7A08fVoyPtqNGTq/uaV0HToKoL1GFrsXjCRoX4BBl/d3GQl28uWsjJS+aYV7kI+dSCpWEBSFuCuTkd4txWowjxswCAs+5Y1Q6oNe2+yxvkAVRIG9ji3hKs6Shu5OvHnz500UCVuy/HaVVhxzYw1pAhCMlev4ppdnl2N85mN1ArRy7goYoi9jMfakwbGfYpXOuDYQwQRsh6YT6bJqitv/YU0IzvdYQXhz73nEiva9Ro+IO6taUQUHjBwN3UGLSMLXlPT/XCftSu3uW4psfqur/bWNlsWVZaKwkr2zV4zDtHb+0fdxoVoLEt8+vNnibbYD5RW+ldMZtht8OffSTi2I9RyYE9Pm8Q5CDROgKEQFIcJI3yGJs1a/syejnncahyJX+fGqlRdRgLWccNucpbw433r2CjAb1qJonU7lvDI9mhxnyZ8msEv/wKrBKw4/LNAtM4kxJPF9LvJRPF5rNsoxwH23zsWYg8A4YbHVcNX1bGrddDVTqyIdG3NvnwZePitcV+NbKkXq7Kh3oJeV/elaUPDnS9tRVvMQNVui/8ng+jVd7hJzGYOm1jrYEKb8BkqvoAU5H148w53sXDBylfM+3Cb1aiKkiWyutWteFEBMt2Kx55kH/0khL3MqF8bNiztBIbwxdiuFMSdidi0nB52f7FnILH77NG+jXbV+9svp9vkYYXZS8HokVulo8wu9l0TqgWsx7UZKlKs8o1ZOpgSVuBDDXJPL7sGGsDRXA6WlXIFtN7H9h9IuzZS/VqVngGDa+BN4j+QIc7kLUl4LlLvAxbJcZlGcauCprTq7/Mquf2ElqtaDZ5j2dJUNdwDrz8kcl4j4h0mZYRV3ZHA0LqX6cDs15I0X5EoNofVtfdN6J3JvWEY+LcUCeAPIyBWuJbMH+hw+8Zsmt2EeHobwbiG3RWoTFNDc+lD8VV5G5V3hALTLWMPva7KO7NZjV/W/oOl60dT5OB0f2dtyfinuh5ZHGMlj7uCqgfUK3PDqlBm1ahX/4GlDlncjVexlvrGNBIORqu6UaNeSA23W6WQnOZLeHHa54JO7zv6m0AM0tP2Nj6PEJmZ2GTZpHE+UfvTaKp602ZssFuXsDwrxFqNqmmlE4CUkJFFVo7l6IqA2QRl4a/GyWOlV4+ab8uGP8CzVwcdNhhW+2CH0YrFKUbPwyzStkkBBgORuBg/d5Tnhs4HmkbduPtB0ZRyek364uH5ht1GxRqd5jwnIJB+vL3bhRVy+fn2EwBye3AH6fMtpifqhlHThJRbLfsIjluK4xWicWuWPr1WiI6I7cEndcKCVHVjBSrdDYsUSZZ7QHov9EqeSlFqqcqnRsdDYgwGeoRgFWsUUMeaB9T1g1reAFiAqNo257Jts5hgtrkuk9zYqOG87FuMLCvbrrCCxjOcEuDEd97pX+Bv/u5ZQFCS3INVI8+WXdW1PnZPRb+z/dhwf2zBCjH+cfKsuLS27g1y9fY93nSP2w19EKaxNWCL5BmyTYWQ5CAEiVhI3OTdhqWN3q+MnYaxvU4MQoatBYh0wrZcCfqXjK8gzJWfkKdPtqVqM+wZqRuPiuGWWKlixCFJup8LcumvHYlMF3C0EtmWzpz5nPKwZyaFnNiT5dqaVqSbyYVQ4UR4eBbtV2K9LPy1M1FmN74p14mj0dpTbmRj2pc48n5meFY2znXl5L1Z8yYW61jP//nizUSbHlx/rQm/1oSHWLPs9Q1JedSLeLyiUfYDV0M/UGHd2RA0wfT9xdXGRuG2EnVFWoXbIax051DRebV3netw2YuuVe56Qdz14mvx+7X4/Vr8jorfjxF6+HgzQYE8NIBsdPPbDXgEzYDlUZDcbUAiJj4CM4+wZhAUBkocn2ndQHk8Bx4/0wJijMa+85uWFEJC29cUOrv0ZafDJip9wknnGGea+QxrFFc1Wa5wr0psdb1nCDRrC1FYvqeV5fv/0WUMFs2/o+ULFgV//EWLOoTxI+M7rMZUTED3wHTODReB4RL9/bkxvGRN/+S7F4GZ7oidkQNtcnRUdDlvQzxlu5DfdC8A2YxAaqG/+PYsoE96UbYGhYMZk2O6KE3LkOQKivIvnDicn52pA/37mP3Yviw0LzcvpIadS0N6Dtuks4q83rF7g1+Oc48Gkonj8F9/souMC0zaoNZ44Zrq3gjaBBShWICto1PbD9q6hWgepd8iO/3uag+uK6QNwjOwAN+wS0JJ3bDdOruBqN6fhYNhl2TWFXWKe7Rq4czC7ogB47EXnOmshKlutp7ftGr83ZueV9YLAmU6vCLQi2j0ClgbQajBNOnjhDAk0jwdvdElGgUyzvpaAoBD+7g1murBFjiYkKGmS8B+ZU5HJqKBhbOARAzXq2lC7tO4rlmZiuzru+VLFBoYRLiTrcNY989fNBVr4U0cr8kZVqd5oLQ55pjmTAfbO7GGZPRZGpNzelhmMnpLEPe/laXsEjw59EUI3JtnOsfxNmHvdepqNlvYy2flfX3afJe13Bd3x8v4IktHQy9NtI52tGFAxoffbD+7XhDYZl9iSR78Uam32xLScrWxGnRSWx0mXBnqlS/AXW6IicBO07LUt4Pjjuj4599vQHw+d4VE6mKA0PS4uqp9h4+B5OxQQedtDRXWfIf17QzJz01kjxyfN1voJwbu/YOwQ64brfrQCExg9weld0TfUSg9WYzCo3QiMzzS5dNwQDGDQlV34/c+mISHdSmV//WTZYb89X9BISsko037qEfP5NqRuiw39wL6etcXA8l/Y9ldIqt/u2KXyehYrqCY68O5Q/7c+n8CcnlwVvQ3HLaIC1wxRKyQLctzLAasOdYEQbS27fUoP/SeYVOnXMwm8MgNdJtAoEbYRxzjWHewcI5APHuL+uiRhL2NdfD1fVjfIOte0hgYkpj3aKX4RfoZUfwosh3eK0OxUBom+k2f3ywtes5wCH9i0H8AxCrJPA==', 'tfcdiff/diffusion.py': 'eNq9Gttu28j1PUD+YWCjMOlQsuRN8iAsiyJZJC/ZRdEusA+qQYzIkTQIRTLkyJbT9t97zlzI4XAoyXa2WazFy5zrnPuQ76qyFqTY76pHQhtSVK9fcfVMlHW67d9Ni2K63hep4GVBc1z/6cRyibNwHpeV4Dt8Iy9ev1rX5Y5IvGWZN0SvrWgtOM31a/Et25k3eK0f86KpWCrMG94Y9vokkwzWAEH4ef0K/8vYmrADb0QTHMLF61cE/tVM7OuCHAALKUpBfisL1q2G/+k+F8E9zSOSGRjeooHn5qGF7B4FsO6zIESYjs8gCwnLG0YyRSrNadOQX375+68BKPvXMtvnzOBFNpKEF1wkSdCwfB2RFW1YsiszBkylZbHmG2CO3fOUyfuMq62Kf6/3zGav2VesDsJpiy603gHmqUJCYo3NeSspwsuOvLNA8QIr1MWTkFt8KwTmrlvXXV0qmLxsmgR1ChCgty/zL/AgqFm2l1qOr5r97iqcijJQJF1xbXj5YPu4qnmW4HMfWSVWkvH1uhVyeYG3+wboXdz5gCRicLWkEaxqWjiJZXnRvjgC3DCRFOwhKUoOim/SLUP7CCw8Zvu1gOrvJfmDXd0zImrOMiK2jCjxCIoXkUcGniMAcFPTTC8A81iX9Y4WZnvQ+NItrVdlUXBWS81oIzxEBNiO52zy1uMCyuebb7UIDuT6mtySN7hcXoaupEjFUr0mIMUFOknNQNSI0Lza0njuJ8vqGo0AQcjEwHSv1zX7JnGbnR4IBQgk5nAKRoOOYVsblyHEopUItqvQlFMxxZfBp2lFM4UkmEXk9v1sFkYEnSS+Ak4aQQtxFWFk2LMYXxWg5/gKYtW2vHKooRVDjCNIo6zBH7JyR7klDb44KY1kcUwkUQqaGxSdct4oJZPrjoRnZw1o39ggTPKcfwdLWj2SP+g9+wx2RcCcSE0rsLoNK1hNVZQ2m66fsWTNgX2apjx54GKbAHRNN6y1BHun26XA+RKsYW45zsOW54zkrAjaVSH5mRQWeA/FlFYVK7Ju9XIyvwMtWPe3d6GtggbERMIWURSR437VtNiwwKE+IfPQIa+wDGlzmxT+02oAer1VPQY53M3vQnJDbo9R0agGsng1EI64s2BFU9aBggyJvf+4mzv6lSUrJmgXo9T+mdv4KucFozU4QpFAEcBk5Ivnsxm4DLhILdCv34HRFhle3dqagwRq8JA4JgaVo1skjzatGM6xUqAp8IHIJd4eaUtQljsELr7taXbxBPQY22ZTxX17bROTwe8YxYZvdiU/l+jkfUTej4kzhNPIA/kUeCEB8jlReg/BiuTFCYVAcZY+XSe3tlJuPVp5e4xsa5ouYQsLkH83679V9GPylk3e+xmWofNkCLp9P6JW9Xvt6k17jHxre0jnJyP5XPmKL6sv7MCdSD0DfV0qB7ZvAoh4rFisnq3zkoqfbg2eeFAFdVc9nfgc2bhwr3oxTy/u+h49VuJEfU2e+qdCQg+Z4gqfI02ME8PX8PTCjmG9DYMyVNB0C1VwWu3hr2yCTH3OZaIG45VLI23Sv0vV6ord2te+BmXmRDLzKXiVs0y9TFIgVpcZVirVVN8E6h3UN9BSQGnQAf3zt3/ASgf0hgRzwN9/GtrlA4I1KQVH3JA1hfaqtnKVvEcmbXcZcBfoddeIDHNL4DyBnBOOwSdVze6ViDq5zKeRs2a56KcZVedCtZiM4pK15DGcYejZDAD4zuqySXL+lQVHtMaFrE7uWf5ImjKHulkuTqCDhMgl7Uq4+Jezu8EG4bOBVSznC89KfHjjU4zPMbtNH+0TaraBphRKwNV+vYZG70rCXkVt0NCxPzwF12fJRjCu8XMwyd28OhUERohJ4NAplcHM030uq8pGFmJtM0a+BQfYvP9AL/BvMZn/N5T7WEKXUzenuPYY4vlcG1N1VRWeRbUsoLXmxb55MX0Zhp7MRF5ufgwPgOiZLEg9QP/Gqx+hg5uXMLCb/xksyJ4gDP2DjYFFVyUyx+EKLVrasrRqga2xnS7ahck9rTmmMatI8e2F8iiZUabHUwpdlfdsQRhUgDmISKRQQftXw7Z4xW4eml4SfPBGh8+Tih9KcL6+h7BhT4QVy8uHBQGztFVM0pxDPsEAn9I9pHc5BWk13iqSN2RGqJCvV8A2tNqQXcu1fNDFHOjD7T79lJTobIZEojmxo61T7iqngp8dlAo7aOmHMkM7DG3TLDzDyjvgHaNFkpZsPR8nbuzIH92OW9KzeLkd5+WILbsMnsWTbxwF6DKeClVqJjh1VuW6GX6h+wk9o/KMv7p6xhfIluIOOMX0NCH/crqWPuAgAClQSdZtK74lrRpbLnXfq9i1+exr3BT+PptQFB02NWbsGkfgblshfUR9lj9kwrcK0A603ScfnaDj6q1STJtl3XgTdIYgScakvrMFWZVlbo3Yk0OMxwW2XldUpNuk4d8x+B6mzZZWDErCboFq+nJ2L0fpqsH4hH2a6jIcU18eLYyX4s387i4Ef6oYFUFHG8KAHHsfpoOWD/oci3/71GPhbrGcoba7MuYQwwCNqouNc8Td0YHuVFNgdWnxALqGVi7ju3geRrZ+wt54pvk/MHgY0O8rrmcNXm6maU53VRJMsD+ZT3sYJI2jJmpksV3Z63hxOxUHB4uVPMOp3WmCti/8TR/flQmeCJizIeUhDQiVH3GN+BPNG3bcNWxu1LVPdNcbHfFjs3d98r27PhvWdei4YeuANfQGhWoRD3IUIMhfIeHLhl+tsJrIwzFFQ0RUqKHkmk3fwc9Q1HDKDlU3jz+td4Auqzam80IKKHhR7huleFvN7TGbVKeZeTgnbgotL8AksN2fQ0UZ9A+qbm7ms7AfNzBQuGd1jhfIeIeRL7HrIHl2tq8h3PQUHsjV49MprWAJJsGdKX47dwdnrWHfAzWAx1lyT5TQHcDb/Gij0wYun0aEh0MAHBCRvziqi8ls4a9QDeNWzJOPIkVZR7zZ8RB3ST7iyTBfc1ZP1jVj5POeZ9JdUPr1HqyPkYey/hqRB4aDRbIxC3BkAKV6xhu6yqGCFX3UaNBJu5Eto7adg5D+w1P8Z+gkOGqS1OxQJ33Vawethajc+EONZKlU6zLyMkO5JOVKQFlvn0kDMqhbWZGy4fouFkj1+mKdfHFewOsCnzZNEfMzYu9hiCV01dITbl88XTwF4xVQvfpTRewb8NnyWg71yXaoBXnYsqLnPpF1Po4HsuBNeGoL+rF11XZ/x3QF6AaKg1ThOBHkDMd6oEEYwIVjhJx0OmZ3xK+Ygh2gv9Uj/OGC0YQpN03Nz48kTbVqPAy/KIO+NEybmbJh0q2WVaodnOLK1C8RHA3i9sKlNd09lf17NVdX2+MXEDImxvL0ezab3R2tB+yOtF9Q9BoG06mYq/ldaKENz2NZ+SsvNucWK+PMOaDhsNv11Ua63z1C8RmlT5teNH6rmzPmq9/0PwoQ3UcBczfVvBl8FfAh+hh9MZWONzlKofYgVdI1hNahzaleUs+Q0HXL3XRfcPzUZyQWn2g7wYqjZwGOgqFhf/AG8d4tliNKQd5C4JSKjr6f3nP24NHHh4hM5oNYcyIe+usa3dxpz/Y0prq704nwKLvxGcJMMELgMMD0umpe1Scs253zIqY3Wv6ZsXLM6/0OP6ILI7zbkJotNB956g/OoIFfZXThaQ0VNfcMSjkU2dAdHQS24Cn2ed2N1dyBJ9QBR2Gvr29DNfcM++NBy4nsvhI/32o/r1Kh9lH+7TSFn46VkB6KcqC4SwmygBoN83XDN73PNC8lqoVE9ajf4me4nloNmlPQ/0/dA6MA9WUdflh30GxZX8uRGwTsgJCcBfF4GqIjo6+Wi2hBooXMpgPE+ONfAKHhY0S+dHhM3La1cUnEljc4acPDAqyJyI42X2WK+Pjps71U1rSPvmbs0TRjeDEMfpckq0sosxGvHZOCD23nZDjUsORnMnP3rAWFUrhmQYtSBZMPbSxRbCqTcZxhC4U1BRQcClqic4wKLVAVWJ+xqKN5nYvwB+LOMEs63xm8NAOek/2elfmekfV8Gc8SVY1v+1t2viLOSA3DrHZpHWSBch5onRFYnzL7I9AXxMzO+VRkOJ4Ou0GnDvFPiKRPTI2exB4PQ+hTR2HOeFoPmJXsJxLWk+beg9G6in+akjVZP0Wyo2J/Wtx+Ix/0v8ceDkHdT4Ix02g76gbHIPo1rTcN/Fx/fcCr8XZAJyk/lE3pf2pKDY8=', 'torch_dct/__init__.py': 'eNpLK8rPVdCLT0kuUcjMLcgvKlEAMnUUMsEkkDCEsA3BnHijFAgXzADRxjABEMMnMy81scjFOURHIbGgIKcyPgcsAFaMImCcwgUAbjAoTw==', 'torch_dct/_dct.py': 'eNrtWd9v2zYQfvdfwbUPlVtZseyuw7z5qd2GAEXWh9QzEAQqK9E2F5lSScqR89fvjqR+2Ulqp2lRYA2CxCSPx+N3930iZb7OM6mJKNb5llBFRN7jtktnMl51GoEQxkT0elpuJz0CP0/Ju+05DpIw+CUYEioSItg1k2TDpOKZUMas42ax0D3Tm7AFSWIdRhK6IrBJvbJv/eKPZLqQwk3acHYdURVJRlOv9hPgTK/0ScLX07DfN3PbvqPa8+ZIz+h40ziunXL0ypuIZ3f5RSfGzuuuE2cwjZUw0SdiOgvUiubsIrys1uqxMma5JqcGsz+kzOQ+1q8c1lma7GJ9DKoVfOGDkJMOo9AnmWCKJyyZ/klTxR4Cl4Vqdru3Xq/aVb2XJ0+emP9vuIrBESOvM8UFI+eSCrXI5Non59uckVMbyySnkq5JOSF6xQgXeaGJ4ktBUztsY7Gjb16fD05JtjANa0QywNi0U6o0ZooJhLwTShmZXJIpKW1Wba9pY/a9AezNGV0Mwst+PW4hiKn2LiAbZbBIee5BTfQvJgDIBGyb8ui1oNvLM9rD7/DSLvjcrdZ3CHID4XwHwnODCBYRw12b7fvkesWh1Lgi/xawY0pUTFOW2FEbxN+FxBxzwTVAQbjzj3NUAXP1imq3pEtdn0wBjE5G5odkpIruoZkRgPE8qIHfhRAQISfEG5HnxBNkQAzhq5JDfgiop+kZlOWxxXdKPBpcBTSoCsvl78/MxrxmVHCxrLZkIAF/knzAJT/4RDFml1xpnavJyUmSxSpQMc+3QSaX2DwxrcEwCF8GwxPJFkwyEbOTJRNMUs0Sa4CClNP4KoA9BSu9Tg8jhh3GaKwFfqIpv6GYc58gKAQ28wy0apU9u4NMj8emM2xH3UxahsWZ0HxZZIXy+g3dzhzgmx2WGaJMRkCr0nIMPje022HbLIbZO5poR65gYOAcA1BiybwzmKsh+dMyMP+hyTY8Nm3zoX+BoMHyl1BvIg9yXhXfmVWEfyLZRJsp76rq5nU3VBt22+igdxbXzAc/6GDQ9IW2j1tzoCnmEJnoktYo8uzCujiZYmDqk9TeWR8mj7oW4aRrAvGPrFkdEG5ntidCLd7NWpLkze+mWFucdObKqaVOjTB9lou3y1ZmY0BftWB5hvcdufruGDv/WoztiO2xzN2h7vwe6prOebQxZrezt/2wxCL7fAWDP1fDz++sYWeDVfz87iq+qsnmiN0KpqL4fIfi869F8UibCRB53eZdRXObmhiyD30ygI7uOeKWY8TMeLXea9kwvq1gWBveseHkRctGNpRvBQN+g0KoTwVjN8wbwUkXvHR6XCCjljzvHxVrdd8EcKmIbpgEqDaBUxMzWOk4eQFWpnEGe8BkYjYvG6OwttpBxZpedsSpvP34hAGOkvsOBKOBoQQyDcjy43jwGMeDUSMzqgP2PLSP5Toh+MfWxXzkhuZhoBH1PFPMSMoAq3HH2mV9Ptq3bT2lMPWHP6hGb77RswrDqgrzf/rE+pIiKkMnPXVum7ooR9VYeUQZlXeXEeZpfK+AjH8IyKPnfvzVBWQ+rqz3cj++W27G+7b3C9D4GAEafzMBGlsBGv8QoIcU4SMKUDmuzY+ow/LQOoxhI4q8hXqhEnbsCRHYRlOIp/huE/WLULFFWPCNMSWpMQMgtkz+BviTXNJYw0EZ8IFSYiWLC83AUmaFSIyzn4clWQBwgCHU3V/v3gfkvYAK1YWAdKdbvwKerKmWvDRvneAIypKq3q95mtpXmlxAsVOFlbjO5JYUii5Z0E49F9GCUYCDqQnk78a8CGNlzmIoLVswbXM89E/cMkiHRSFi+w5M2B0teGpIWChW5RlZHEVIpCjyFEsXfntVn9iLRE1un3zkVLk3oM39BicGaIoH7q27WdUDeMVqOe0O2juTWaEZKHImvTqlvrHsB3WcnQg7DRMd/mm97IUBpqOa38pssxX8U2J0BJgFuSgUakIbvdrutL5NsC3z7M769Shc/1oowB0QX+G1roD1hq8ZX650kFBNrTyH3mnfNAPtNf5YuueRH+aSH+MTbA8K0jt1VVAn7fCYDwv5oBXacyX7VHBIbrSUNAEnpighm0kmnmmSQu3Ysv/JPaxonqfbyJLeXZhcwyhAIxevqSAfsRSAZNdcr0Aq6lq0aoEkSjLot+fp4IgzRXvJidGj6vuTty056oo5bLJINZK/LVlmQxxihGAOuJK0F4ZHYnOe6AzcdrA44Dqyh+/4cfAdf0/4jr8qvua01rW95XH5oLMaR40XIH9RZEgZRWsKshk5Yjbf9ZzD7jLphcPh0H85/PWVe4AH9uQReUPffSfn4kSdnLaevTjHt6Ji7Rh+Udi8svqo3DtMMmi5wAOasYYHOZPaTgrWtPT65HcSssHYJ57p9NtjnUj4HaHw+2Lhu8HwL4mm9x+W41aT', 'src/__init__.py': 'eNoDAAAAAAE=', 'src/models/__init__.py': 'eNoDAAAAAAE=', 'src/models/main_model.py': 'eNrNG2uL3Nb1u8H/4bKmrDQ7K482dWgWJjS247iw65ru9tNmEVqNZka1XtZjdzZtISUfQloK9YdQSgmNY0ppQ0hMW0pnKf4wJv9j+gv6E3rOfUhX0pVmdmPSDolXuo9zzj33vO+VF8RRkpEwD+JzYqckjK9f81hbFiXOtPpmhCEdFDaajXEeOpkXhbaPI+5dvzZOooDQ1ijyU8LHx3aSebbPu7PHo0D04DNv9sI0dp1M9HipAH792vVrN8g2/sjt5fzLjJzky4vfOSSeLufPPOIs/soosvORF5FptJz/wyGjxT/DCac0zcYZA4CQHiznL3LyaLr4GwyoTu6TzG7ODCf5+eIvITkBdEDNyB0TbLf8KE21OHFHOC2ZuFmfhNZ4nA3Nne/1gY7Y8t1wkk2Hr39X371+jcBvY2ODPRwuvginZP/gbYJQSJYgAljQxcfk4PDeIZl4y4svbYLQyeniU47AqEG5QbTbfXKnT/Z0sv0mvPToM+vkPWRIoRjp1I5d1kOh8uZTzz3Tts1yGsME3RxlY4BAjXTuksTN8iRMKSHjxH1sncBGAkO8wLXGiR248LLDZ+IGWxx5yV/OQsY6+m+FeeVjn6Nu+TFKLCcKYt+dDQ+T3JXxlusqMVe3rRN3J+ou5IJde8v5H89JYE9CL8tHLtFePgEpJv7y4vchHwqdVfbYJ6lWME0aVFtLMYy1VxAzwsg9I0hdJrECS18CpaOSMdF2Z16apdpMiCwHMAONJGGUkQdR6Iqx8L+d+5l2avt9MhIzvAIItItGCdQpmgLpfaTpOKfUeG2kE9dPXTJiZDm+DTpy9+7DfQ2Mzn40yn1XwEUyLMsDplqWlrr+uE9ObFhqEI1cIMqJwrE3AeLcU89x6fvIYyaLbZJEXprHbqLpRgFOl/oAssGAANvZQ62XYoTOEn1tAKMFRrCHZq+gjA0Rb+W48onbw8ZPHvGfPzz5iBzeX/xyf5fsoZE5c73JNEsvA4VS5tvByci2UKUL4o82s8T2ws1jA6RH25SGbPaJaQx0NQy0ECtg4BCAMTBMHSl0Fp9Ry/s8nKgYwYDD6iwUHss3ATrIyJ6JK9ZAynMqUcPNNA82dSOLNLZ3ugoYI8saeeNxSeQGvuYpANk4ruEFB2qlmRunxWg692ij6GhMSd3MCt0zK4w8kJLUmboozJo0W8hqg0KU9MB+5FonbmaXU5nMi9fhpu+Frp1solHD7aBkDM3BYACDQNnBO7nbt/rEDUf4tCMrACihgEOGQyJA7VZtH6JPC9vjo9+2HaADgVO4FdTSMly/hmDjcW6PNi4BnvR6IBiM+uJZRoaNO10YU28SRN66SLdf75PX25bTnMeBa7QVaCEa0rnN+K6TLfbQsId0uGyzaYShFhS22ypxkVaURRYlCH08C700RmDmhmmUwJTsPHaHrG3sR3b22o6AM+zQD7FaZu6aoljlTSGSFdUQrRvHdXcuy2ubNtXcMBPoymBGEbY3MaDMNwdDq6ymenO99C9Y/8x2puAfnDiHf2n0LDwXCExmhyAxdGify8MhZTf3ZdIuV7lq+/GUojENkJXaMNZpOYAsiTAwCGODv2isr09scLZD2eJWJ6GzP2Uz7TiG1Wqm0a+NOdrdNo/rNjt9nGRWKyzs1bpg6nqrwU7cCcQHbmKd5OMx+NxNuurNfiG5XIX0VfOqGGUA7bSsA4musx0c7V4NU8FAGaZgYp3U9eBCGGYFXpina2CgknVpNH40WRcLDL0iErqWxHW8eL113PwmKALzakhgYaauluc4QlxelFinduKhDRAGA+2/gidMdgCFkmGr1tJEJy+h2atfAiLut5hnOb4Hio0MWpH51MUA/gRgkYI8UJADsSHEHANdvwxdgWtjZuWOzUtQI7ZArWTdm3Al4nYuQVyHXNQpXovIMmzAxM5zMuYBLSyssAiCBw4zC0I0THixTQ4ZaFnkyxDS0flTjzhTb3nxi5yweIFMlxcfknR58YRk08Wn4RSSfTtzIB834T+dYCL7AQmnGKm/fLKYhxNyClmtJ6Zniy9CghA/zMlrd+VwG7ZUhBOtduAoOy5rEBRhFcJOE0JDzbtgNMIxRlYPeQVcZyh6jGP1IO2xVQhCwWAeBzNOyyyuyoygWiXmQC7Fz2LeLfJuh1y1AdkRQDIVBSpVb1KkGgVgGxyrou+vwFNnYsyIFsMEI6mc4hRr5FLmj3bJSRT5Ug5vzYZYjZCZfIKCaaXee2iGZ6zodTSQSGbxtO+e0lydxWn3MARmwVothj3qDISOsi3zWA4dwVDErp1pJRUoa5h1zoxGXA2Bo7QSubxSS05mKNVRKTFtKt6UEmTiUKj7sKxSaLMKF/syV/RKxpZ+O8S04EcWVSRASYvh+HYQW6DcBtYepPkUfqdQinXImlxHwrIMjoxq9pCtRW9owmqEsvR/nxfSI2uS2CNRbmI6kdpYSWxXhuE9GzanWxlkatizaul1/astfyj2rYq+8lYlQ3rWa4pXqFxih6PQ8r1HrjajOVRG3iQDlimxEe+5SZSKER2MBvvIQEO4NTBuwZ/mUnXDncUaLXKux3eYHcWFSfdCusDMC6M8ZYyX2VyUBSk7RbJYqxAysF4IIgEjNZP8jGjVItLNm+agJvxoEOrFwZoOUAuHtg5gV3ucPAGzUmG4Rke3p/qcwXQanV7tGoMz94gXwphTN4F91wDqxNWwtFRZiq7vNvVf0MOFjgs4be0TT29OwMyafKfGuiEZ7KrdoSCcrdcBM3xEm/oM8zEs2wsqebLKvClZWTBZPkV5RXw+YtR9e5yuKuvsFTFe1AEoloaXY6qz2+AAqjKd3rkp8sCjbfO4tKCr9LliRUuvPIQYkG7R8Aiebpk7x50KzvFX2MksRMXTi2BDPJnHugS2dhw0vPJPQKD1/bt3f7BPDt7af7j39o/wSGnxjB5ppsv530OSJRHE4mmUJ2CeJhC/O/qrQt/O8hFoWasVpZ1lme/WgDeBzRwODHijgj2NsnRoXtbEHoruQjnKvgM8tKnglqqldo5qCGKOuesYaE407RDTrgNaTtZpJdfjz7qkm1QlD8ABmfpxe2WvCBnPvGxqmVULVRV0uUqrHZkGyhKr1dKFVeEZtKduZOoV0ua8csCxzqq/mjIdinIIeh2YBBTD3pQdyADco5IHxa7pHd5JZTuvYDfbTCMoMtsKlUVk59n5kXes6BNFTToASz2KQZdLG9ZNH44VmIokosxeRBKh5Mf6oUJhYeOyiK8KwJuzFDa5CxjT9W6QVVkrNMbCnVLqTSXxlMaL3WuZRLsVM2cDcdSuzWDP+QFOWSy1MlR7eqZ0U+7FDuUGCELa/SQeD7EFVtRpDSaXM4W1BNokogqa63UtIBaX8e6KwpQMixsnTKkKIFgQ5cAhk/BCtNW6gg0jD3IklL3ySKwNJOPWtlhar7cjQWacb9FCFDfOj15bNqEVfMKsAnrF9sArP49R7sGsSnjJ0V4hMFvSKrfkslDTbm6B4WtEEWX3zdLbrc5MWKrlhZOurKRPclA5XLt4Z974cs4W+CfAqAMxdhmi7ueVHr7ELyEXDzIFpQvpDAXR0omRsJmmQmFa/ZbsQaw1nJdyQxVBYG0rFHrRvfVtHF4Hi1yLVAVdvBrZEdxeITMtUhcOXzKvIvXgPdWgQYoYzHois2XW2X+7f6e/JxJRZeJFF5XDqqzSxUpHpet4aYj40HhEgZGHHtAYtHjyFY4ckpL+lSa2TsMs5bYiOqg20bCAMUgZHKxiUWc/K5nf7pNtswa2tZZTz/4a2aiQUlV9kBfaeJ6qSGu7iB2usZSi+q+CzsqSCnuuNyxQtmZy/K0nxm02QW0OWnglKrT1aqLYdHHpj773Cbu2tavwxAybritOWLqksvvkA+zTdjcACCV05r511bnNK0u97y/e3yd7Pzw4IHfuL3794D6k379ZfMBu3NHLsa8s2WaVSbzl5qYV939eBtqN7SouCeNvHzbM+/f7T8DQ2WCD8XKcNHSb7Jn8CjLbZC3A87+QZNPlxScemXh2qMvD6RXlvfLisnzCN1pePCcvn0SLp2EJZzn/U0hPEPUWAsWW16tv5SVmPkI4Avnokl2tDSfL+ec5CacekuQsnjpA+OIpkLi8+BXo7KMpNoVTSPedaQQ0XXweTkhAJ7FKCT/VxAnPHZJ5y/mLWKxwim8ZOUFY2jsPfywtJKtYQqwjNMp0fbwfrReJrVgLt9mGD9jlJPwG3gW6M83PlxcfhEjiM6DuQR48PAe7A48Pzw/p/S7m1ugF8ZdPlvM/nxOfrR1vsAOR4hxW8vaSJ0IPZPEz2mG19tDptbrvj2VNT3SD7Bh8k/BeO1A0Xc5BPEZff0Xf6AV3ZLNRueKq7X39VU4W/9old2G76NRPYOtg6jndPYjscdUvQlLwGS18cYUfhl58xM+tGY4tkx9WZ1P7HKV7/oLJwrbJBoB4LOfPA6TmaTjFy9m/9cjis0BajB+d0UCphZOVNDUH0985dkuuNdwgrxnkwAO8lOaqPPPwhEyjxacgjvRfdi22kAaV3IIQVO63dkYDbGlbkjCj96/vK1bD2MK22Qx9fQxXiDdegduZ0TPJ8xUxSHnQx73kJUKpbxR9KCMPvWri7kpWtTZyveJP7dyW12oYY1YEBJc6EK4BPpeylnbw6ivpaATBMVE/o/JIRHVjnd4F94trJfLVcCE6nGKdfT/Tu9PbaycBDFfp7BTujKjv3svbNRswa8UEmZboJR8pSWlRkeo+WS94m5VLUX8jwD4yovTTr4z4R0UCEVo84Xc1x8ezVPmQoOAm/QhqKH3zxAFIAWXrd0+rPl5Ak3e4vPgYzBjs7LNYyc8symyfoi62VfokoVfs+VbzW4NeuYRGGFqCrcfNYGnP7GRUnr4Dt3t2MgE/3us9OsOn9vMhHqepZ8kftLy9/5YWnfzEdbKOj1mCfDgw3njjjUrKThUNTy2CvH5nd2qPojPo+enPhUwjUHGBTQCtfEIj0vPQDkA7Yjux8Ws8PsjAVvD72OoChFTTmyUZ2msAv3MvcVNau1KkQxKBRwj1mN1Ph5kjO7MNB6IgtzykR7LzGHrc/y+iKbG0UIKVTL4V6BTLpQhBhC3qtQCQV+kG9v90ifIeRPG5palJ1ms0Wzi4lXDWQsdgcAnxosYH8b/8cyhawai0NONHCRYYdHuEVidzLTSPYqrU0ri9ifyVQCgvk4hOeY0STITTpvGMUfLEOo38E52ioaHLhdaWY/4LirYG7Q==', 'src/models/unet_1d.py': 'eNqdVs1u4zYQvhvwOwySC1VoDYjSBkYAF2jiLHrIZoFu91QUhizRCRGZUkXKSXvaVyh67KV7bp+ge+yTZJ+kQ0mWSP04GwswYInkfPN9M5wZvs3SXIFK8+huOuHG20wICCUIMZ1s8nQLs3WSRvcS6j3f37y50B9cuMh5fMvql/cs2XynFBOKp8JbTifTSZSEUsKHG6a8JRFi9jaNi4Q559MJ4HNyclL98Zbw4cvH33Eb7J4+/8khgITtWCJdkPc8gygVgkXaLH5BdKj8cavTlRPwhl+/BaL4lknFMseF3X9/lT6h5cYtUE//fuKwTpVKGNq8n3VcidkGVisuuFqtiMTTLnCxiu5C9CCRCxq4sA4la7+c4Re2Xa9ivl14dO5CWqh21aN7shaKfgyzsAAaoL//aKZRqOBxpYB4FPYbnJIMLsa8ZGGtjdiXRcZy4swaNsbGU3jVPHB1c/luefWD8cnceK0DAR4QLrJCwatv4SwwLGmJZkxEHnLY5wUxqHXk6p5cl7HTh41UItaRRt7u2Th9EPok5tVlKnZe3D1nvX5DXbhnOf5dSf4bW2DYpEJMtsCFLIxjLm4XnqOztkueAjkLNHOM7wB1alLvYXY+DAtADwmgrRyQgI5L0McPjhbB10k31yrQ12cDKvjjKgQ9L4ZV8A+qEBxUwT+gQg9/frQKARCkr1V4jVcbvnz8w6glfVGCcVHmPaeGRQkOijIfEuX0+bLXgQqVEojTqeCk7+BICVlePV9Cyhvkd4CLLGgD92MeCpmlkvUiOD82j+08YS8KSOCU3tcluWpMZTtST5//Bp3yw/dEX5EeTf9raAbH1qwuzZdcRvocTTpcE3U97NGkX0OzW5WOJPmCuuvUvo+T9GyS7wqF3a6DueEiTA41HLP126TaaqIHjE2aP4R5XM8Xj27Z2V0QKUdzMgrbEanX1R/PIQn1DQWCQ4JTHT0vj/4K6VqyfBfq22tZOwf5S65ImGR34Wod5mMjwylclgIxgT94hFDEpX0Ik1Tc7icOwGrDhEQQc5jJUJhqeEQD5Kea1M8ulINRlWPkwgUMzfVQIYErEaUxy6FTPsrhoJky9BTiDC3WgwRhnq1juzdu9paDA+603Ch7cNPSSWxmXLtWt2vC6CgMNWH0ThvGN2B8EpsDQbtW90PC/FEY34TRO22YwIAJSOw7Q2t1hyEsGIM5hYumZ5RnsKuE+w4xZFJ3ErQ3GOElqyJMHri66032TjfyRWMUu0Rl01oysq3Qw4FvJ1u3VDfKNWZ1PyCF7Wvht6AYHgvU74BicBg9AEoHw4XVmRR2tAraglI7WOWSCaqHQe8AqDeYilgtSWFnIhar/XJZ2OxczJkqcqE3/Q9q6xaA', 'src/models/blocks/__init__.py': 'eNpLK8rPVdDLyEuLT8rJT85WyMwtyC8qUfDwc3MC8Xm50sAKkooyU9JTUdU4gcVQlCWWlKTmlWTm58UbpsCUBafmpDnCxA1dAAALKME=', 'src/models/blocks/hnf_block.py': 'eNp9U8uO0zAU3VfqP1zKYhIRUrVoFlTKBqTRIM3MAsEqiiJj3zRWHSeynWFgxYo9fAE/wBew5EuYL8F2muYBM5ai2s29x+eec8KrplYGTK1ouVzw0SmWEogGKZeL5YIKojVc3ly8EjU9BFLG1zVrBYa75QLsWq1W3eaSiAJualURwT8jgwsuDCr44NriruS6FYY/15QIBFrLWw33X7+7HSWm3976TWnB7r9841IbIimCtLj+hULNWUtEPLudYQF5ziU3eR5oFEUEXOa0JFKi0BHUrRmdDqjsJteWqE6CFxGcR/Aygs152I/llm4bVEEYn3DD0Tt7RVy5efJukgRO0lxxbYJ0qHXLvnxt6zYseJAWrNcgUAZjcqHlGk2RhtUQxrjcJ4f1ehv1p7yqGSZnCguB1JyF0+aiVnCwykwUGEqy+YRkv/fzdeMdJ5hqOT1tRghPfSbmNu6AlpzAsQXE7x8VbEH++fWTzC53Ichdb+7t9xTeHKFc0GZUnH7bfwagpmt8i1fvrfaNIBSTd6rFeWWfrMmoj2RoEwIvxiGDJ8nUT/ug58xQGm4+ufwMabVWfCSKHcN6Nw5elyuLZbmkTv7gLvTWeSuse/P0ZUNv19V9x/azCk5YETBeJWN/utKJz07QUYVzAPxUsS5Jg+km8ypPMHp3/uOZg0t3Eezc3/YnCx8mmvZQXmjX5Zpcc5Y9zp2aGW2FplXS1zybmmuF/AtyF4cR', 'src/models/blocks/bridge_block.py': 'eNp1VLFu2zAQ3QPkHw7pQqayEnsqDHhoUrRLEqBNMxWFQIuURYQmHVJyhE4tOhXtkk/IHwToFo8p+h/ql/QkWRLtJFx8urv3ePfuaDlfGJtBZmyc7u5I7yvUGpgDrTv3nGWYs7sTK+YcHFnJZ+JImfiSaB2eGp4rQce7O4Bnb2/vrTw5/ff1Zsqc4BAbzWUmjZZ6BkaDNtIJUGIpFBB3ZTPC1CJl0ZRZSkNENzRcJBBFUsssiogTKgkgESzLrXABiPk04nI+GY5etddWx+ULYQkNOxz1YkgRrnEwaRm24olUVRB7OhdXudCZZIr0OdXB2InUglmypujrgn0Y0eBR+rk8ufAroZWQbYuJsdfM8nWHRQC9GF5jL3r3GMhRQMFYcDFTzPpJDz8WsEjL+9+odAeAZbn6Bry8v0Xv8A3CWRanSPHnplx9h8w+3OoUpuXqJ/C/dxCnEvNzGAzguFzdHcP7i9dn8PFDufp19q6/rKef9Ha4lOKaDIZesygSZtTiOqlzZyRnKkKv4BwXwpu9NwpsTEQulUnWYqvBVILTuk9UAEb7rezbyAA6aE8UxmmuL8kogHpvtjFtcphrh4MXX8RmGx1jzfVMkhVYj4YC94AM4WXDSCujQvlTf1KK9Qr4c5cJFCEWTChMJnA43tytAgsqvGIOvVq4WMpY1AmN2YdSppL1K9h4FAcHMNoeXPXuQ2VmZHiIh8IBkA4+gMeDbv4+RLEgjcUs0zPRYYJ1XZPmh6JSg2qq2zTFpsSYhv5nOvUvjllGPjUmSlwvTNCGjKu/Pzcb8NTgMPwf0nGABw==', 'src/models/blocks/attention_1d.py': 'eNqFks2O0zAQx++V+g6jcHEq190scKmUA3SPcOKIVpFpXGKajFPbKRUnXoFHQDwFe+HAk+w+CZMtSZx0ETlE4/F/fvNlXdXGevDGbov5TAcngQjSAeJ8Np9tS+kcvFPl7pX3Cr02mNwwRPHW5E2p4vV8BvRFUZTcPMoevn7rlcCqpvSaXIWSeQz575/4Efz93XcNH4z3pUK13QsKPlNytYMs06h9ljFHMA7bQiKq0nHApspajEtfdFnbzzW1siwWfVwc3BFC9HGQDoyJpvVlua5I0iWE1eopOU1D0aDGUQsIkqRBzVFPq4v7ux+arrQEMn95Ms0QFrWzHpV02B/bglFsDB6TnA3QnrmA5xz2ytIhc/qLSpNp77U1n/5DuSQMu9gZ+1na/O8qTuHYX3PYcHhD8JNwhazVcHWuvGuCnWJh1aOEURBVPN4KH8+SmEETB6qOQ4sj0vs1h6tb3pnJYF7fDiHPuqA1tPmCPE+nkN4j8c8vv5KeXiw7CG8luto4xZbXHJZJTMwYVsAmi1/AlXj5L5ozO1/JE2udHEifLsMNmcZPE5+Vx8v08aVrCqJ/OOjNuE2rfGNxeBWM5PEfDf02VA=='}
SOURCE_HASHES = {'tfcdiff/unet.py': '644f7f46965aeab0ffe1e6ea4bb6ab1277dbb84e14a1c601dc26be828a8f5c1d', 'tfcdiff/diffusion.py': 'eb24e8b8473342181adf2db04f30e53d7b20a4be89d271b1a1196df825508aa9', 'torch_dct/__init__.py': '83f5d262f04a65c44d89816046382f4d148229882bfdcbd9d7ff6d6a4faf1d90', 'torch_dct/_dct.py': '233bc5275e1ee0424c777dee5467065a9f1200f6ad2286ad220d170df38ebe7a', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/models/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/models/main_model.py': '6a37b7ad6fae246cf0ea30bf34ad245d8c3a56a53a52346fa256099cc2d8642b', 'src/models/unet_1d.py': '2e226d29ab33f46b3603c5f2ac21d93c67c7204a0f4f25e216301137153ee4a9', 'src/models/blocks/__init__.py': 'c75b66b49a1c4d73c2e599ed9650501454dc6188c78b07f3169ac0b08673d4e2', 'src/models/blocks/hnf_block.py': '4d48ecdbbda1a6eb2424fad580e719aa3fc40dff3f3a415a68aaf9ac1a1f1ff6', 'src/models/blocks/bridge_block.py': '13404db64ea2908b686c1b7e0589db8cbb44dc4bf99e90a5621139f7ae4dd4f4', 'src/models/blocks/attention_1d.py': '6a51885a14b70f39b383752d328423fca6e68bfeb32478063cadd81d92d27842'}
SRC_ROOT = OUT / 'embedded_sources'
for rel, packed in PAYLOAD.items():
    raw = zlib.decompress(base64.b64decode(packed))
    assert hashlib.sha256(raw).hexdigest() == SOURCE_HASHES[rel], rel
    target = SRC_ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
for rel, expected in checkpoint['source_hashes'].items():
    assert SOURCE_HASHES[rel] == expected, f'Proposed source khác checkpoint: {rel}'

# TFCDiff chỉ dùng timm.layers.to_2tuple. Shim tương đương, tránh Internet trên Kaggle.
if 'timm' not in sys.modules:
    timm = types.ModuleType('timm')
    layers = types.ModuleType('timm.layers')
    layers.to_2tuple = lambda x: x if isinstance(x, tuple) else (x, x)
    timm.layers = layers
    sys.modules['timm'] = timm
    sys.modules['timm.layers'] = layers
sys.path.insert(0, str(SRC_ROOT))
sys.path.insert(0, str(SRC_ROOT / 'tfcdiff'))
from src.models.unet_1d import UNet1D
from src.models.main_model import DDPM as ProposedDDPM
from unet import UNet
from diffusion import DDPM as TFCDiffDDPM
import torch_dct as dct
print('Verified embedded sources and imported both models.')


In [ ]:
# Load đúng nhánh EMA như notebook train của proposed; strict=True bắt lỗi kiến trúc.
config = checkpoint['config']
proposed_base = UNet1D(in_channels=2, base_channels=config['train']['feats'], emb_dim=128, out_channels=1)
proposed_model = ProposedDDPM(proposed_base, config, DEVICE).to(DEVICE)
proposed_model.load_state_dict(checkpoint['ema_model_state_dict'], strict=True)
proposed_model.eval()
del proposed_base

# Giá trị từ config/base.yaml trong TFCDiff repository.
tfcdiff_config = {'diffusion': {'beta_start': 0.0001, 'beta_end': 0.5, 'num_steps': 50, 'schedule': 'quad'}}
tfcdiff_base = UNet(in_channel=2, out_channel=1, inner_channel=64,
                     channel_mults=(1, 2, 2, 2), attn_res=(250,), res_blocks=2,
                     dropout=0.0, seq_len=1000, norm_groups=16)
tfcdiff_model = TFCDiffDDPM(tfcdiff_base, tfcdiff_config, DEVICE).to(DEVICE)
tfcdiff_state = torch.load(tfcdiff_path, map_location='cpu', weights_only=True)
tfcdiff_model.load_state_dict(tfcdiff_state, strict=True)
tfcdiff_model.eval()
del tfcdiff_base, tfcdiff_state, checkpoint
torch.cuda.empty_cache()
print('Strict checkpoint loads passed.')


In [ ]:
# Suy luận đúng hai pipeline. Đồng bộ GPU khi đo latency.
def run_proposed():
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    prediction = np.empty_like(clean)
    latency = np.empty(len(clean), dtype=np.float64)
    scales = np.maximum(np.max(np.abs(noisy), axis=(1, 2), keepdims=True), 1e-8).astype(np.float32)
    with torch.inference_mode():
        for start in tqdm(range(0, len(clean), PROPOSED_BATCH_SIZE), desc='Proposed EMA DDIM50'):
            stop = min(start + PROPOSED_BATCH_SIZE, len(clean))
            x = torch.from_numpy(noisy[start:stop] / scales[start:stop]).to(DEVICE)
            torch.cuda.synchronize(); t0 = time.perf_counter()
            y = proposed_model.denoising(x, use_ddim=True, ddim_steps=50, ddim_eta=0.0, num_shots=1)
            torch.cuda.synchronize(); elapsed = time.perf_counter() - t0
            prediction[start:stop] = y.cpu().numpy() * scales[start:stop]
            latency[start:stop] = elapsed * 1000 / (stop - start)
    np.savez_compressed(OUT / 'proposed_predictions.npz', sample_id=sample_id,
                        prediction=prediction, latency_ms=latency,
                        normalization_scale=scales.reshape(-1), data_sha256=data_hash,
                        checkpoint_sha256=proposed_hash)
    return prediction, latency

def run_tfcdiff():
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    prediction = np.empty_like(clean)
    latency = np.empty(len(clean), dtype=np.float64)
    with torch.inference_mode():
        for start in tqdm(range(0, len(clean), TFCDIFF_BATCH_SIZE), desc='TFCDiff DDPM50'):
            stop = min(start + TFCDIFF_BATCH_SIZE, len(clean))
            x = torch.from_numpy(noisy[start:stop]).to(DEVICE)
            torch.cuda.synchronize(); t0 = time.perf_counter()
            z = dct.dct(x, norm='ortho')[:, :, :1000] / 3.0
            y = tfcdiff_model.denoising(z)
            y = dct.idct(F.pad(y, (0, 2600), mode='constant', value=0) * 3.0, norm='ortho')
            torch.cuda.synchronize(); elapsed = time.perf_counter() - t0
            prediction[start:stop] = y.cpu().numpy()
            latency[start:stop] = elapsed * 1000 / (stop - start)
    np.savez_compressed(OUT / 'tfcdiff_predictions.npz', sample_id=sample_id,
                        prediction=prediction, latency_ms=latency, data_sha256=data_hash,
                        checkpoint_sha256=tfcdiff_hash)
    return prediction, latency

# Có thể add hai prediction NPZ từ lần chạy trước vào Kaggle Input để tránh chạy lại ~2 giờ.
def load_cached_prediction(filename, expected_checkpoint_sha256):
    paths = sorted({p.resolve() for root in SEARCH_ROOTS if root.exists()
                    for p in root.rglob(filename) if p.is_file()})
    for path in paths:
        with np.load(path, allow_pickle=False) as saved:
            pred = np.asarray(saved['prediction'], dtype=np.float32)
            latency = np.asarray(saved['latency_ms'], dtype=np.float64)
            ids = np.asarray(saved['sample_id'], dtype=np.int64)
            if pred.shape != clean.shape or latency.shape != (len(clean),) or not np.array_equal(ids, sample_id):
                continue
            if 'data_sha256' not in saved.files or str(saved['data_sha256']) != data_hash:
                continue
            if ('checkpoint_sha256' not in saved.files or
                    str(saved['checkpoint_sha256']) != expected_checkpoint_sha256):
                continue
            if not np.isfinite(pred).all():
                continue
            print('Verified cached prediction:', path)
            return pred, latency
    return None

proposed_cached = load_cached_prediction('proposed_predictions.npz', proposed_hash)
proposed_pred, proposed_latency = proposed_cached if proposed_cached is not None else run_proposed()
del proposed_model
torch.cuda.empty_cache()
tfcdiff_cached = load_cached_prediction('tfcdiff_predictions.npz', tfcdiff_hash)
tfcdiff_pred, tfcdiff_latency = tfcdiff_cached if tfcdiff_cached is not None else run_tfcdiff()
del tfcdiff_model
torch.cuda.empty_cache()
assert np.isfinite(proposed_pred).all() and np.isfinite(tfcdiff_pred).all()
print('Proposed and TFCDiff inference complete.')


In [ ]:
# Metrics shared by all model rows in the final summary.
metric_cols = ['SSD', 'MAD', 'PRD_repo', 'PRD_standard', 'COS_SIM',
               'SNR_in', 'SNR_out', 'ImSNR', 'latency_ms']


In [ ]:
# Minh họa vài mẫu cố định, có cùng trục thời gian và tín hiệu đầu vào.
examples = [0, len(clean)//2, len(clean)-1]
fig, axes = plt.subplots(len(examples), 1, figsize=(15, 8), sharex=True)
for ax, i in zip(axes, examples):
    sl = slice(0, 1400)
    t = np.arange(3600)[sl] / 360
    ax.plot(t, clean[i, 0, sl], label='Clean', linewidth=1)
    ax.plot(t, noisy[i, 0, sl], label='Noisy', alpha=.45, linewidth=.7)
    ax.plot(t, proposed_pred[i, 0, sl], label='Proposed', linewidth=.8)
    ax.plot(t, tfcdiff_pred[i, 0, sl], label='TFCDiff', linewidth=.8)
    ax.set_title(f'Sample {sample_id[i]} | record {record_id[i]}')
    ax.set_ylabel('Amplitude')
axes[0].legend(ncol=4)
axes[-1].set_xlabel('Time (s)')
fig.tight_layout()
fig.savefig(OUT / 'example_signals.png', dpi=160)
plt.show()


In [ ]:
metadata = {
    'data': str(data_path), 'data_sha256': data_hash,
    'proposed_checkpoint': str(proposed_path), 'proposed_sha256': proposed_hash,
    'tfcdiff_checkpoint': str(tfcdiff_path), 'tfcdiff_sha256': tfcdiff_hash,
    'source_sha256': SOURCE_HASHES, 'test_samples': len(sample_id),
    'seed': SEED, 'proposed_batch_size': PROPOSED_BATCH_SIZE,
    'tfcdiff_batch_size': TFCDIFF_BATCH_SIZE, 'metric_batch_size': METRIC_BATCH_SIZE,
    'proposed_sampler': 'EMA DDIM 50 eta=0 one shot',
    'tfcdiff_sampler': 'DDPM 50 one shot, DCT 1000 / 3',
    'torch_version': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
}
(OUT / 'evaluation_metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print('Proposed/TFCDiff stage complete. Continue with Comparative_study cells below.')
print('Proposed/TFCDiff inference complete. The all-model summary is generated below.')


## Comparative_study: published input-length protocol

The [TFCDiff paper, Section IV-D](https://arxiv.org/html/2511.16627) evaluates
FCN-DAE and CBAM-DAE on the first **3584** points to avoid downsampling shape
errors. Every other model uses the full **3600** points. The paper does not say
3580. Each output is compared with clean/noisy signals cropped to that model's
input length. The same 5012 test sample IDs and metric formulas are used for all.

The primary all-model file is `summary_metrics_paper_protocol.csv`. Its
`metric_length` column records the length used for each model. The notebook
exports prediction NPZ files for reuse, but not per-sample metric tables or a
separate Proposed/TFCDiff summary. Results from existing DeepFilter checkpoints
on mixed BW+EM+MA noise test generalization, not the paper's published scores.

Upload the test NPZ, `best_qtdb_mixed_bwem_ma_3600_fresh.pth`, and
`comparative_checkpoints_bundle.zip` to Kaggle. The bundle contains TFCDiff
and comparative checkpoints; the fresh Proposed checkpoint is supplied separately.
The notebook verifies checkpoint hashes before inference. Failed H5 loads appear
in `checkpoint_status.csv` and are excluded from the metrics table.


In [ ]:
SOURCE_PAYLOAD = {'descod/backbone.py': 'eNrtWF9vpDYQf4+U74AuD2cSwi57yfUaaR96VaOrlJ6qVlUfogg5MOxaAUOxSdh8+o6xAQO7m416UftQHnbBnpnf/PcAy4q8lA6vsmLjUOHw4viI6TWZl9F6+ORz3hDxybKfVDySLOc0VRTXx0dJmWd6W1RZRsuNY1jMo6HIqFy3O2m+Uswpij8+OulWgSYhrWKWh8VG4yEnl8BjRd3eK54opUI4P+b8MYgJKqXv3KvjIwevGBInDBlnMgyJgDTxnFNargT+nT48qbuWUl2iKqAkrt9xjGktUpTllyBAhgUtaQYSSkEMQY89oVB8NiRqrLD8B8oyxlchz8uMplpX/wnYai3dKfUzlLkwRPeMKsV6X/yaC6bD8hOP8hilKr/8ksdVCnv8ErNsry/GxiO9s1RcCrmVmeQleio2InnOBIQpPEI6sLlfXlr3/iODJ3IeWEBRXnGJKB3gbOYsLD0kFLirEwR9zFdAGha0Rm4KGEhvVryeedsVwyOLRmzNkuvMtDI9PxjnogI2fcXFXxXAM5AhVOA6p0ZTqAtynnISwIVaVFZYbHPX3QqieSMqR4Jv9YZgnLTUrtdS56JfvGtivBw4uARZlbyDURttGl0DxT34E037IUkYh4OyiPEwWlPOIcW6yStpPVXoI9pIMrG/pqmAV+XcWAS6Zbw04tChUZ1KxYn7vwO6mUtG05Ebce8GhdCS7DThlARnW9WwI+buLoe6rQjI7iG2Db+nMlqjgrUv1rSA2/ldv8eS7bZfDfVfUWyxnnMPkrYF09tObFxdZ9NCaJTwnPPAw2T1o3XFH8hC54ydMuqqEYIEzplGVUlc44PCtlIXg3s1ZVOEL2s30GWSrrWdp1++Xn9O8+jhwPQsMKCCPYPnrFkcAzcPMUupapqHpuMozRKWqvauc0xrccOEJLdDB5hjaocWs9mF53zodVm2N55T0FjV5zI4nayFWR7D8n0JSQqRfO96WyEHxo4xL/diLt4E8/u9mBdvghnsN/S7V4He7c6GCHUJA0yGF5RqvNCZvBN0J44aFnTK/cyFpDyCr7gyBJzNFi8oungzRbe0QLu+cDhicUVVF6+3qaiKSpXUrdUNUZyT0g2UWMqD0hu1mobXp0WBYyJpGEjtjrrYFKk/Y5uVafOzhhhD0bJagdfMe1roFPjax5H3YROWkFak1+K2CzPReK4BvGuFes7cXxwu2oo6aQXuEmCarZZz1kXLJtUd+HPJ4hX8k/77qiHAmommM8rOBj+ZPoKxXM2qnNMXhC3Ovv9gteQDyqGRj6PEfoCBvq9AeP28UbcJ29uM5bFlv/U1GcmaZMnYwk7c4B0tNi8mmCeQHpQwCUZYLD9eTDNkLM5rdNiTN0KWQLOwfumQHk6JJk6BUeXATrj3PUONmqokf4ObP4iqvjF5N9IYSPMXHEi3OJDu4hvLCw48Hk0gME3i/2PxL8eiKWeMwpaPBo2k3Zz3Tdd/KYDmbLCVG1vwXyF5aaDD7tb37dbN324y8hxVEG2bFRFNYfrRpA1XHzti01sdfOfghO+IzfD0zAoy6IueHVd3y6tbO0aNdrox674cHxOus3UyQVWQsBvjlCa18CbNYayDaRhajYbgrN4zuAwip+nVYcTUGcNpBmHoLJfOuzDMKL7yh++ueikcTKiHB9anOQ51VUzbo+VEfadEwvZ7pGolCRE0K1IISypheTGfo0dCM6Mug8Unz3liPM6fcArhy4+X3SPazWLFMISou6m0pDzmJJh7gXcZLIZUmy1UQ4pnVaggMUC1txnMDSdFybgkz/rTw2DLfLYlyDhY179/Ax10Yb8=', 'descod/diffusion.py': 'eNrNWluP27oRfg+Q/0BsUKy00SrWFjlADKho09M8JUUfzptrCLREe9lIolaidr3n8t87JEWJpC72Xs5pN5u1zMvMN8OZj0PatKhYzVHZFtUjwg0qq7dvqGrjrE5v7XdhWYb7tkw5ZSXOxfgvJ4ZLmaXTzCpOC9EjH96+2desQFIuY3mDurEVrjnFedfN77JC94jnrpmWTUVSrntoo+G9fSP+ZWSPyJE2vPGO/vrtGwQ/NeFtXaIjDEYl4+ifrCTDaPiP25x79zgPUKbn0F4MtOtGQ9i9wGm8zzxfzBngeJmPSN4QlClVaY6bBv3447++eeDTbyxrc6LlChhJQkvKk8RrSL4P0A43JClYRgBUyso9PQA4ck9TIt9nVK1I/FPdEhNe01ak9vywF+cbfSA5VEJQ3ElzeqVG6BzUOwMUFhihHp4k3MCtBOh3wzhnRs6aJhEehfHgta/RV2jwapK10sfxZdMWl37ImacU+lOSFNIko/t9j3tzId62DQi52M6qhyRJGk6qpp8npWwu+o6FyQ3hSUkekpJR8GWT3hKx5J4hR6/oCLWIhwJ/J8mOcDxMVZGh38aXOS0Jri8DVCaQVUQCiqPVagWDOORSHJHrjwEiZSaebswwgVDVclAcIy3KGCF+hHphu0riXKQeTgGHEC7lWqoNM0juKLi4a3F28QTx6OoKrUKFvn82lYnGmyWNDT0UjJ6r9PqHAP0wZ854Xifck62ABXkC57Xyu4/eq4cRa8jhqtX8K9Z7JlzUmk8FzXoq8jhLJEAA2rGppwBzUjashsn8sSKxatvnDPM/32iJ8UIOaesVSUyEpg5KK01068XWjtG5XApsl5/6UUFuCVOoRLvQKSJ/3A2tF1t/bJt8BQrjOL0FBk2rFv7KfVJzOwQLxyVEy0RoBF1k/CQd3XG/sd62P3Fe3UqlUQhR4wxTnUkKqmuWCeqrwu6Np/oChGFzilf+3KSkqsm9momrCiz2ojBwxmzW19HW3R6au5ons7JEr2P8kgbfn2XHmhxgdyV1smv3e9ixLqUPLoM+grvU8k/NszWaAuaxnCNJWn15KiJnlMnJQuMw/R1KcZ62ORYbV4P2rEb9FoTuvGPC0a/omPzCr6PffISBTRi/JXVzCvXEgp2PWi+p6yr/LK1QRiUFLdvmxfplFjwZRM4Or4MBBD0TgvRDTVJavYYPPrwEQBH9HhDAKZE/nccTEV0xAY7Ck4hoGcsyqqFaOCYmW/UDk3tcU8GpmoDFXjqxFiqjAN7kQpmg8I7dkzUiUHHkYCKSRnn9325uL5cXkdivVSOHQXKP4CcdP7bgfH+P5/qWCTuSs4c1grA0XYzSnALTZtCd4hZ2F+AHw+O9I+GUs0KYy+4dwC5LWh4Q28uGgXPSW0zL860UyaZVJB0Sk21Hu4JIKngpYKcq2mLC5gCJqnTlnxHlw+SC4DJJGdlH88p1HE2z23IkPQvLzTyWhVh2AZ6FaeqoAOIymnJV9yTilKyqyK54lOkHv7Jt4ig77PtTRLbhW0Aqtqdr9G/bNmfiiIDUVKnWrXbvkt6NPcruWKHgmjhtj+sqdComlEYHZidZFOUz8256I6eUTkX+GMTUKBA78ratPjihx/VbpUDrYdp3cn3FlCQj0t/ZGu0Yy427guQYi3sP0687zNPbpKE/C/I9hs0trshmZUBWZ5Gc3Ms7AVXffhGHBlXkOqG+WSwgN/x9tN36kE8VwdwbdAMNyBP8MRydP6DoNvCb1zdrd4khAtkQGnMJMSZo4bpYJ0c83IF4R8t3gekL3zrpNn8MGEe/7SRr5SfRhGmOiyrxrkWVHoWWBKljMRy1LWbaTiZZ3KmTaRwre/xRDpxWaMb9X7urRZYcapzpCy2VDQ0YlS+kQfwFwwItp4GJRj1Pme5mnmN+rNfOVm+9s2EYz76Tcn2y1XAOKJOcfifeUZ5BOfoLbO7ybKlG/Exq1ugRC44G9lOiobxahR/hZWyqH5Jj5fWhcdrvMJtVPX/TUhrIacnaRjnedHN/NyjdqQ/bzjWhEktLCAkY6UVQPXr2VdyHD9HKtzlCkIJ7wehkgeQ2wXKJWfPI28G2BmqxHO7J0fPXIp2D5TQ53e4ShTBFtIQx93CAI5kHUg/EE9dylim+vx5zgMbTBV0X4LI1QNQfTxA3E+hPjutitFpPV6MauLI3BSreyKZAad6C2bSwbhamKG7Slb2T1Vbyqn7eKHR/nKftZD2+kuP1XYnUMlnWqT1PJdF65AuR1FLQ4vKYAzfX0da9b5zPbItPhz06jgK1uvEGnj5GN9vFVDcLS5srrH1fFxz6Kdr6hlj/PMSKXOGEcy4PzYNzpvrjonWK9rqydUHjM1itD/tOvhH4Omy7nqFDZAOX2SCTIHKT4H3kZsHn4O/BV01ik0krjWrBqmSo64yLyVMlYXcUFNnOirAtKWAsvOn8OFE9QhgHz5o4O03E9edxn5PpsjZVDpokqFMuWuwP7yl5mPDH5wBdR1aNtlgbuGwyYjcduVM1Z1e4dby3CDc+w5hriDvx6+syVh07bcWykjmPOhdp8/cgy7msn074GV9o491aUy+h/tBZvg8Q1Oa7DK8nqj6lzb1KVgmFDrjAI2LznhKfV8Pp2L23gPP+4tyrqxtfXV/49infSCKzZBSf5ZLGYulH+XfaU+/EkDWU06J4bejB+pj43aPsFDMfu07x7YJ+xx5GavPcekVTX9fvkh9Xn3d0xCVeIEjHlApVdeRbnzS/iC7Pocpn0eQzKHKKHg1T1ZG9c96IGF+dFMeE+IJMGmJDBdAySQ4n23MS/0UU+dRTjXPT0N0VKKtO4HzSFYYj+NGok+bFDxJF5mtR/Tc69IJ1qsZnVzHSZRHIiQdcZ8N5H9Bc4frQwMvV9wfxNF/qdQQ0Pcv+UF59Wecf3/7msd1/SMoXvqhTtPEq/PTpk1XwSae1YHPRup+v3uKMPUDPL79pFwmh+pZZC7W+HqSLuxIX4LAK11h8DaobFIpWSGTRSkBC47lVHsSS7A1rctfSmjSyhl7PsYQEuBFSt+pbBDAzwxyHaQ4UPVwPCNhtBT3k/wu0BCvLbHGV3i2F2KIGU/RVMCzR1YwA00pS4P+pieYasOpRRd4Ysu9gTsTgWeCqRY4RO9NjRbxuUPfafdVLEr3VMqZ7QxYkOM4EU3KSiGtPPdVoGX3EIvxriJi8xtKdpo2GTCFn9qMN6Shzooux+2JV3zDK5T5rhzH/BbHsSu8=', 'deepfilter/dl_models.py': 'eNrtW19v4kYQfz/pvsNIkRpzcl0MhOOQkJqEpI2O+Kok1bW6nqyNvYRVjM157YTLZ+pbpT71uepD1Yf2c/QjVOru2gZjG2z+BqKQBAI7nv3N7G9mmd31XmuJx8sXe+wXoI3xADoYuTaxb+Co8x5OieVhl73LaO87JrZodC3yvZ7jNuHURbZBqOHAd9g1yY1jw4XTx64jpHAfEasJ3UHY1mh8fcM/UgynLwRuiNfzr4GYMaGgi6Xs4z+kP3BcDzxsU8ftWs49IApelzd1XacPt9hFVAmsglD4En/yse0RZMlwzlsmZC30Gbsj2TZTjGU4duw7tS3DqYU81pcMbdcZOL4nwxHyjJ7muH1kkQfkEceWf3r5AqY9DMc2ENPA/mQ4NDxyF1wDZ/aAq+MdVdpXzN904PCeO6h/bSL2enl1LsMFpj00YB+f9K+xabIR4xIMMDfYsBClIdSRBkk0l5oBJhN3QdeJTTxdlyi2ujJ0BRuozM23saVT8sD0U88lJqatigxohLK172LL35dhgETfrX2K+pi9f/Xq9h65NzTqhj+oz4ZaKimj7kZCMRmGQAkBQCuCkmiP4WIysXcJuRAykwn/S7SP7WAi4zcJqdA0JhL+xz0bue7aJ5YZ+o3wAdPFcEyYzZUYfBRN3YsGgWlLDKw0yZHQ8lbcIfKkSMzwlpR0jAxqKSEfjaAUd06GXGx4E25KCEaDHnfTWKKUHvnAW3E/xX1pIMuKu3KCPEPmsYD5kiVeYNhkYa3g4QDZpm6SPpWGjJpDwihaKkmhBoA9+PPf3/756+df/2jChyMZrpjFLKo+TqrOHCRpyK7Ph0BZ8sAPeKL7oej5799/+W+i53i3LvZ814Yhd8Heyh5jbXD+rv195+QSYi3czZ0g2+ssBfqWQB0kuMjbnaNyyE61LUU0JLYnhXnwK6glGZOiYzVLIJ42LGKzCWY/SyyiqZrVOJlnuJ8j0OoioOOYDzLai0GehXg64MqygN9sGHB1WcDq5lwcSPGIjc2v0gfGbZlzhT9V+FP140g4Ho88TrS8QNEWjZT8UMmYYrOkZobK1JHUNhAs0zAvwjxtA7GyYrzrD5VVAZ4SKJqIFE2EiiZiRZsZLJ1Dbfl5pbGL80pj1+aVxq7NK42tm1e0Rbm8/Zk/PwYP8mJwKuqFIlBbkNG5lF4v5urSmNUNOrrw1yUxDyw3M7BazWLazawZIo+d9fUlW4GKyeguA5cZm4un3Pr6Uu4ysKvLws6kaD6iYpbNTL8LEWVFaSzfwGWSWX2NyWw55NWlkWentAKoClk3V2JL5rRUNuPX7Ykr0wu6Ulx5eh2Hr7ycdFILLybGAz3wnH6HbGJZSA/oL1FyY6Poa8FBo1Zq8s7jn2qOjcFzwKcYkP0ZxOKnWOSKcIglbQZ3vJgtlSbaFOYfKTGA9Zo8ZW06l3FzfPXmj9iqXytub8YCZMFvOekhL22NtQsgz8FerWwx9t1FnoNdre+q17caeR72LXd6mPL7wVbhrPyuTU3w25G0p1eYTy9lz7R1qxP2qpHvKu7NpbyNenyLcT/5RH2m52ToWBJkeTqZBmMyrFUc2ZCChJm1yx0cSuHbyJNr5SQ46lGvlWbLBZ/PIVitrFpQrc8rOHCxSQw+yjRdQE4dz8IkmpNIhciUVU0GdpWS1ZY4LhSeNmh9EK8fZXB8T3wQsz1ZL87kpPYYpNSKslIrTEutMN2Wl0wzU3um5uqpyddzp/JyD67etd814RzdYvB6GEzHiJSvJ5EeFmZt5jLOyHU5WqdTfCVq0yzfZrXpSHs0tQtFcJHgLR63eSH7iNEqmsKYHW277GLshodrpbJSm8arZSg48s30MF8zghkhu2nbN45gRgLYtO1rQvCcpZJZ6vRY09uHJzOS0Vl/YOE+tr3ggLYzugbQYOA6yOhxt1ImgE2We6LLjnsE8TPw3ypwpbAXSnBPhh8V9ivDqS/DpQLveYPPpd4qTFCGK4ocWUh8wRXw4/yX/AqQKmX1TUmJlGsOoZhZZPrCSNYtnBx/E+7TUPApPyfe9S3rM99uunMsn4shiyVlm13KW5HvOdg2mDdcOtJ7dnJyAoeGgSmV4TVLP+VGuf4le1aryjpS8DDNwlo5TYy9iQVAPjzZK38TW3wZaxlxlmavT4xvLkg1JYkpME1YkrM/l2FspfzIRgyfDdhxA2q7boC6TvzqivGz76ZiPjrl93NVlCH0UOq+Jhnue4TNCoQC/uQzoBabG/iW+Vt+F9d+UjzpmPGNOEUWAPJcNYfP5qr9V8SCtLFZfN6ItZXHsLbybO2ztU/C2qcXt1nF0nzJeQEHLLxgO9/KbQEPLVJpTa+z2hcXmj6qPuYqt9oXmja71jq0PeMB3YoyildKjZIS3DzuYsN3XT792th3Wf1jY+/ecW8pdFnFPC6ZxnXRqBRC7g/kjvc2cAm7Xrxtqo3ya0VVDw5UZfZpt8nNSn6HtFSvybMPMoSe06nQxGqw1pXr4/SmrbjxW6hLbbiuRFiVs7hXyiyk/wddlqx+', 'tcdae/dl_models.py': 'eNrVWltvHEkVfo+U/1DreXBP0h5Pj2MnO1Kj9a6dgDaOomxWPASrVZ6umal137a7x25H/BsQ4oFbQGEFyQrYIGDNZZe1FRAvIAFPq+URiQckzqmu6q7unptJtIKJ456pqnPq1FffuVSNW/YLvC5fasEPIbse93lAUx4GJBySezRwQ5/s8Yy55E7IE0Z4QHbfuEXeTngwIm+EwVHoTXA49cgOC2AItm9P0pAFg9BlMfkyT8fkfkyDZBjGPjTs5h1qSjpJx2HcJ3sMFY5ZQFhKqNcRncyn3OuTwXgAHT6MeG2ELZ1B6OfyL7Rm/Mf9KIxTcshimly+NIxhueJ9xwcjvYTI/rfYuxMWpJx6JtnDnspYj56wuBgLOCTMFOBYOya56dE0ZYFJduIwCiepSV6n6WB8B9CgHn8owDa/cvkSmfUahMGAggb4b5LtQcqPchnypSBCdThRb0dAHIU4823qH7gUnm/d3zPJrXtvm/dYMqYRdO36B8x1YY+g3QsPqLd9BCsYsbth6EEr2DvPkr2Jl/LIOzFf5y6P2SDf+CaQnQM6OGSBS2hC3tSRUhDlgBVCsLgkjIdeeIwS6bDoCCZ+dIJtQSTVJAMOLbI74aNifnh/G8hjk42tbvfyJZcN5Q4UwBgc8XLyyUwy5F4KRphoWcA8J+EPAaAkjbnLErtnElpAba/GzJusmiSiAjx7NaE+W233c6xWVlZK0MZpGiX99fUkBQxCQBeXhYRdp+vXNq/fuLH16qtorxqvG9Unyrhj9Jp0zIjYN2IcIGWkiSn3mZOkLALbXe4n7VKZXFMftAIxgGygG8eAVsEY3mEdoVZ25dPBbJ5HxvSIaVOC+8+cVU6jTaxhKCfHt6gENQ7KOCFHlpIScCmlj8TJhJpysMS/T/INIF8lq0fgRO5qbSMy4EHuBYYnHiQDbIcdlkUQ0hxEzchghzMOO91uV5jRLlXUPMuQ67YVdWa7ioaHbVQIZrXniGmUK9/OGa/Yasg3C9Qr+spn28ja8/FKIOqxh6yClZKJWTqJA5IhmVsv7ZU7M3rviKUO8w8MyCgO7E/D2W6xNCGUHFDISkyFNQJJhoQBK0mfuxIoIbDzwK8Eicbi9JgHzK2pBDUABK4bndcw8B0PCgtM7AINRQO0rFntmmycx1oDGkxiXIFHRzQ86K/19oVAFUAYcPnSwKNJQu7fvBsmPI+qIk2KkGzoSaZzGx8KDMTJAVN46jhGwrwheNCYBsA25VBuehIxG8yCMETTjZ6SbMSt7XiU9KvUUapQU5/cB0cGK7V4otxbRpGTcEKOaQBRJSQ0gjRBwGkELmnYKTW/yU6Ow9idNqEwtl8Ep5OoiCBMopF0IL0OKeQhwhOyUq5rpTNjXckkYrExHVmIUoBZu1NA2NYjM/Z1CgxsRMEIos6Acc8omtdJr02uwK+aHA+OnGHM3gU5kJFGGtXlWtWP6zgyCo/B3KYLW114mTiCQjgaMaNrVg000ZB1fbJKd7uqUvtYt3xAodhynQggB+PvgDMpUF5DuCeBSPsl/wbU8yT3RBhNZlKsH4HtPlG5bptsuIo7sM0iYVRSToZs1szr5x7TJ+VeErWZZI9CCMyWU1Sxig+Jx4I8BSS5q7bJK1BH1MgZU6x/700CzIS7cRzGxgr6hBBU6xhj/RKSAwZre2WlQieYp4EvcDgIUwGyCE/1Abk5xLaJbl7dsDyO1GWbVJ6ysWqAI0AK45FTIXw5p65NuCn0V4jeEa1aqg4TJ5MRURA2U8FI/NZZl0dTNZrxIJn4xgo331n7An9nxcxVmdXpNPki8urZvZY9nAxidbchBL8fdPcJaQEZD3iK7KGwh4PDudCJIB8xmtb8FLUhrCbpw48O5r5IHXlekE4C88qk2q34Yvlh5r6i16XlicqRRy2p2Rwz6ubEh/LZwU+JORwiLKabH0HsrvLRFqmcQwQJt/G4Uvo4boueeyoCBtSD3INyxWJrW221NilYyIlzwxfBkEK1oZeOJ2ibXZpNCrvtcgVE2S6fbSSUXr/IueQpyyiHFekWOZ2Rq5LXCuoWucngeHszjI8ppKW7NE4vvnBQ3jQlP3+UVWO+B5Xa0KocM1bwmLGCJoNZreff+eX3zp49/jmcbPyQuyb5yYefnJ+ef/TBCEYtv+7ZBun+/WDN2q+Z1iz1ADtYKAJ3iFhaGwhfGPMRvMU2CxqvlxVcElE8MTu02PVqgijaQaxS44gTtFHFZjWlwXi1RrGZGuSh22gbxZBFIuXJ2lhNwmHq02z1AuLyiG0YaxZW4U1JCSItvQsxKj46fuhOPBUeyjNWv9zFynzVvZzHqvzwWnfOi+jbqJ98p+rPFkE0/cbBWEIyJ0Rhmz659I7/aqusEuameAKFDYTcHLW6CnUTAsY/UDtWKNiv7nhFUe4dquDfdt1bNGWuuFszlqzwr1w5hEg1qtRZeZVbUdcobgu5UmxqFQdxFbILCMF2YzrT5+HDoq9WhbTI+Z9+//hn750++s1nT8/OP/3wX9X+QFwfqnrADf3OJOCYwAyxGXaRHzFT+5DmqWejK4Eb4lurPbXoyaD2NiyIS0K9NqZFfvuLn352+uj5o/fP3nv85I/n3z59evq3X33aSK9Z3pJ7o3ZP6exs7xr5BZNwAqx3xBVTmV/trWtlirVvyCSrWvUMfeCFg8PE3iqzGGRgtLLVimDNMbGIn98q5tbk9xGy+CO6GRhbtDHQK24BJYiaXFFKZF15k6FFfmtryi2BJmwb2ocZlwp6gDicdheieym4OqMxxA3YmNMfnT376M8fnz/9+pN//Pi7p/9uSpaXcM2+2hVcbmi7TOao/Q/nP/zmJ3///l+RiT/49cfnyMavfev9fyqGasBUPRBCUVfTdfaNs2cyvlSNFaLT8kYRjDJVbLbywSo9dzsbWmfWdZbbnIvtzaKt0XZG1KuNAXM2YBb+aq1Oc7GOWq0/8RALPXhmXRMG7JdslWOad+S4OdhZDrWa2G30/neZWrX+Qky1pjHVWoqp1kKm6nrE8Or2qXiTWc5yeH+e7FOYKuub7LN09ll19kGCgRH7tSEzyVeFah7/Woshac3koNqcqQMVOtbU3mkAWQVAFyGdWmGTd7q+fEwNckvbkbmsaZGLwjSDPS1yAYwWgiStrq/KKZeFAk0yCTaVdNLGzWZUSams10QKaokX8a8imr2ojxVBoDeNEb324jjUWxiHSi1icBX9XmGA8/JREl/uvSyIWtLGmv16GOrVmdMzUaiaBXvzaFNg9bsnz589+ssHsq6N5MWoI74QAQUzvlHQCkpl1gZejqDmq1Ul1enyB37J4uCfAuQ3e9OrXf3YILRPu7LKNuZdV6lquUS2JJE6TI7wIAlLcmlKHdROU3tV3bs5Qx4nqXY2BRXXcKHXroplybWLpg35YbPg1/Rvju3s2pyv2OZRcvmCZHmSzgx5877/k7TVFpxtVgDZnMm9zZKjWwuB2lwCqKkV2+cBVO/CQOGCsy0BlKU1TQdqqwTq+kKgtpYAytr6fwBKWzL8uirr+Za8NhR9N2Zidl0iHcXM5eJrpmQKdLOtqYJ6YxlQXzamMxPuy4BV3BFg7sCnvM+z8/unffXnHomtoVe9iBLi/wGD4iof', 'cbam/dl_models.py': 'eNrtHWtv2zjy+wL7H4gNcJHvtE4k26kbwMAlcbpbNNEWTfb2Dr3CUCymESJLXlFOnOJ+/PEliZSoh5+xu0rTxDZnhjPDeZAjkjkYrPD14w8H+BuAIYRTcAXt0Hf9r+D86g/wzvUiGOJ3ivZJ4EAPxbj2LHoIwlPwLrT9sYvGAfgIQ8f9GvjgUzCBYQBs3wF/QOTBF3CBQX34QhHhxHa9U/AMUXvMPv7nV/JRexxMKMBXN3qY3QHXoUDjBBV/ryQz+edOpkEYgQj6KAjvveAZ2AhE90nDIwzxB/j7EQt6HwYT9kmbyQ441A38cwb9yLU9HVyTFgnWs19gmMAOcVdQBxeB/2QMdTAMg2kwi3RwbkfjBysIJ7bnfrMjN/D1//74Ayj6Ggf+2MZs4/86eO9PCQlC0xze4gFA04B0cmVP7hwb/765vS6ldkVYxLzb849B4BG+GH8CrXeeHeH+BKUx6e7s8SPEI4t19IE0OvA+i2xqLmFwxLSsg3tqVEgnFHzojZD7DXeAotB1IBqYOrDHkftEdTA4DKE3O9TB1HYcbHSDQ2RP4GHrlAnz008/pVI9RNEUnR4doQizFDxBOp7Eio7so27vTb9/8vYt4TCGF5k6BTFzz9jaQPQAAXqwpxBod2RcOIuRO4EjFMEp5t1xJ6iVEuMynWKqeCTwiGLaBAZTJUMJ3DZsU7K8iXWHe/M88GA/QaHL4L64V96N0LGgQ945eUmIEIrYTJ4Cb0ZUySFTTK5wjiVCks4omRSY6/8UsAEA/wOHT9hSncPMQMzBgJud5tFfYH4KPrThfIrdf0SUps3xAM9dPNCtlmQYrZRCxpI1LvYgtpxiUxbUMdAk+zJaJWiCxaUvS+BjY9X4iwrysfXy3y1t3ipVF8LxBH6DkqpilBBGs9AHc2LKB2v7SqmB69+Gv19d3gChhTj1FcsFIxz6Zh5ljQW22Bevzo/5yBnDZMCwaWk8/h2BrlJF4oh1VABiNPBcH6efQxVYPCSGqjETPhJlXp0byzAt8txTtNdjuYzjYobNVRl+u2WGO6sybGxPxXF+IH4ppFjtM7ZunVgL+WGSH50vrRhY9EjmK1aZszAka1l/qXYYRf5UQZU6TOF4WltwmSKel7E/awses2Z+N+8w62K40F0s6i8WdRiLeoxV4TJXZ9bqGaa/jxmmv28Zpr9vGaa/UxmmZvTv72n0r/bBXpUPFnK9lAdaS1p0pUlvlufOyjwbW1R07WkTzQSr5ga8kPQwfUeVI6rs82Rz4ZZyhWFGIWZO6Z3LB92TzQXdVdjurMq20kirOaonWWkAXspQ1hTIqgVcJZydbDCcrcZ5Z2XO1UGtBle1pFsotGWjWi6eEbwDipmv7moicTnkpXWYy6tcGcaBcDpiuhs92b7refaIOYCG3K++HU8Nev1u65R0L35qBT4EUQBmCALbf2GlQlqDizmhhW3McFrS1lpSWxtrSMsM4Um3qPZVaXMLTL/Tmi2tkw5Eecvqb+Uznfygt3ZG2iU4rxyrjrnD3O8v55V6N072Ve87zXm13ndc7Tz0T9hjw7IobxWG+d0I3cVrze8vcJfKuuNhe9287yvf2wzZW9X5DvP9FwjX70cVcVoIhThaZ4OhAINb6c4OjYVNAS+J63wbwSD7dNZlO0JOuq1yuHjXQ23AjrluQONkUcBpCB13TEYZ5ZeTRslj+HpGtKAh1TIm1doy2fUgp2+6gYgNIBp8pr+/xDtK0ECQPbt6LLVJ6zWM0qprlVZts7Rqm9vqkHnLtBrTXL9pkupuoV0egNvfhr+dgmv7ke2RcoJxTHwzgfSsttUqizrSXqYSqsUmvhayeSvfZbJ5T3s1skt5cB3nre+3VS77it5Km7jPJg9h9tF3+b5b7bjdLbKrVUww0U2xm2+YgxKX3bbsW+egJABsW/YNcdBEqWyUendhjYZnlyXB6P1k6sEJ9CMqHtkhzXGAPZ2GgT1+IGpFGAA6OPbEaBcPru1/1cGvbXDbxr+QCx908J82/tbBu5kObtrgD9IwI1Af2hhQB7fIDnQK8TdCAPoUDCNp5rHxttWOiVuBiyCWyJlRIXG34PLiF/7MBoEZIgct7mee9yJu0LY9HJR9jEpa7VkUQH+MtRGihO77y8tLcDYeQ4R08AaHn+P+8cnP+KfRaW8iBM/zVtg9zhuGVAUko6Mu/0nP+xTFDNFI1QWK9DBBrilrl5QnSZCKh3UKWc3jVxZi3giw5wJ0910AY5P8G2vm/wB8oCe7HvB/PwDGENzit+SsCRWLx1kdBywUQdsBz5A+M8+cS8GR2adxOyaKw7cNUDDDqeTZfgGR/RABF4GJjWe/+Ic7tkkkh3/OsNgeTjRZZRYcmRrMSxbvdaoLVQOxwIgsVFjIjNFSBraKTlROtRWlmDusFLNRSqOURilNTClcSe5UGqquca+QiZZZharWoPTUs9WsQJsVaLMCbQRoVqDNCrRZgTYr0GYF2qwrGqXsY0hZSCexuPyKJE2+3qWGnuidUOL0FxwdgTwLWTlqZ5ykH/7okR6SOm730lbVWjDHla5al4lMrHM1Nfz0yRola4yFFlXDT5ZVvqI686PxN/uRLpbIeqjfarPLzUI4noUhBgc+nIV4lePD6DkIHxG4D0JhYZSufpIFjx3+230ivU1DF+PTt6dG//hN2zB6PaNdfsJK3hRLLuzSTrp6+bZ5rrcRopTwSmtwG85gfnswG0hCLre1dy3AhtowCh/YLn9HkfwevL/+eHV5fWndgpuPZ7fvz65ATczF7kUaezZC4GZqk8H6BfvOEE41emVabIXEYEcj13ej0UhD0LsvuuRMHE9yGE/WHB7gSeA6hzqI4pg5eIdX4TDuh4ao2RSGmsyNDkinrXbCg3BBGGlpJ/Sw4SWvMzBkeZ+bVJfwL7zW1RdoJRZA9ENmnlw3c1GeA/c+ZQkL8uziKMa2C9mRTU4rstvQdMDvQBs/2D7mSLwEDZOQxTyVYycJfpHQTM6Nfz7WcYwFxheBjv30dcRAaUEEjibQ9pNbuH42iDbglNxixlwtxZzY8wymPa+FyFljR0u1z4QDnVL7wpENATigpZBktJLwXUcHDFfSAr14UKUH7rEBOxeas/8dMv6tWb6c9cs204LGS7bvJT+/vptsIK1d/HpmWZcbTWsXzE52wa0FVrbl1ku5cSnSojdJFi+y9iUczItjQbVHV1Iv8+VFfHX3/T7j+udn1xv1e0xfe4xvJ25Xuf54lPjPeMQ8iLzi+6vHo9QYVbcJpcgoQUYxMhqJlsxSO7OWfITATFeGBk6CWA17lQ0LMTfkRo/4dQ6GsUlBchfnMoj4yRR/lWsXQ5TwLsutwA0q4Aal3CA1NyjhBim5QRI3SOQmHfK7mes5fLzFR2ynWdFYLBnRagdddA+kNCJrWZc1KucLUZVihsiqsGCMJRbECaqsWl1Wo4IFVMACUrOQM0amOtVzybIoHityYqPHJMhllStFPHoLjYj1d3JpTDbGce1kIlysM7G3nB6l3uIeJUypR/HiGhZXzmJK514wfqycUkg1jThJ6TVqe1zX8a02mRIEtB9fRqxokQsiModl4QTrM+2GPEbxgwiQ3rKazc828hwnwpXeh6VoVUxD1ICVSxTl3EQGE6THvG5X0DXzT5mVYhRJeTJ4tr6cvTlKS00yt02AzmL1cnpvsvTKr76gJNOPssLgBc80CGiFNb6pPyNPqQqT4rbUQ3WY4lfWl8zChOY0krDPCwG5MHmwdHY1FesPstueW1sLLVUR5Nz6i8YQIsV+Ro9SzpkZJ8CYWWGaTBLLJ3j1e27WeecXPIHJwDnT/IOY42OjBQ74G4M+avgzcu6aUPbKoSyxAY0PcRysClGcaVXY21h8vGQ78HZ22iXy10y6vutJ1z4499LTjyGklOuXTjbua2UXOJtFfzpJMZthklXWVxwoGWP6V3pkNgosstgaY5bjP6mjthj+O9NaYaTf26pgUV9Ik44wgtg5FjZ7PO3+bg2/fB6/k6a/xCQUp1dxbjsAh8mc9lCVeBaZCBclL4mGPPf+i8+JF3XkxHOLAO4U88q4LCgMQ/G0qnAKXDFdXWnmWy/4XF6cfb/RBwvXZN6ddFhdWUYf7ERWvnl0p+TMGXYLuumynj9IOygVFsmpVtrjHfkTCpmCv6zQE110p6y2c/TMUnpHR6ZwmVCK1anA6grXL6VY3QqsvhKrV4FFJCaJq2NmfVfC5IuIHGq+R6dbhadk1OlUoXVVunTMKjRTuNMoRTPK0TCOeo+o6DTsCKHKbViLk+Q4Q0qA2VZT45+UwHRqwHRrwPQUMNxneYMoIfFwtYSsJaHr9CQJs61djX9SAtOpAWPWgDEUMFxC3lAn8GFlGMXDZ8pjZ8itHXnUTLm1K49XR27tySPVlQXtCfomwHJrV9Z3D/wD5El0ZIV3GVBHBjJljXcYkCkDGbLKTQZkKNVuCGW/i7O1JoGUXvVGC+HAwkAkvcymAppLpBpmtg9FOslDtDIzvI4yx5T2w9OMTOiNMutU0KGJp4pOt5pOvw6dXjUdmpyydGrmKjL5VlOsZE2VwtTk6giqymxqat0aw6hKeGpqJrO/cmpGPWpF+bDKeB1y3kJah/NDOhn6rWqvLt+x0uTiJhc3uXjRXPwv9pdaztL7MJbJyQs8U8t3WJm0FcfPPucLGclRRkWRo+C0YsHhhroEuqsS6K9KgET3V2ZhZSWsNAzF51OyqF9qVTu4j6Rmx4+KqmsY5maKGOYrTGNzW5WWm8rWKI8oaNevkiiR6xZLlMh1ayZK5IVKJ4UU6lVQ6nOvmm4WKa5ePaVozOqVVdTWVFRdqekbzWSwmQw2k8H1TAb/DwHSg3w='}
COMPARATIVE_SOURCE_HASHES = {'descod/backbone.py': '97c42788adf7c9c13a95e8d908fe421120688fa71da6c225b7875ee29dc64fd5', 'descod/diffusion.py': '6da57f789534373b93adba8236921974c5ad56fd89d965e48071ba18dcee10f4', 'deepfilter/dl_models.py': '428f94431b0646ca0364bd50f4f3738718cd4156cfef0a10e232d96ea78afae7', 'tcdae/dl_models.py': '09dbfcb5bed1af80b2efe0af7944d25994ad13c14759430740865edf708db8d6', 'cbam/dl_models.py': 'fa5adf3ea55ade80e9fd29527cb404d79f80f28f9ba7bebdc1be7fa617739a1a'}
for rel, packed_source in SOURCE_PAYLOAD.items():
    raw = zlib.decompress(base64.b64decode(packed_source))
    assert hashlib.sha256(raw).hexdigest() == COMPARATIVE_SOURCE_HASHES[rel]
    path = SRC_ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(raw)

def load_module(alias, filename):
    spec = importlib.util.spec_from_file_location(alias, filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[alias] = module
    spec.loader.exec_module(module)
    return module

torchsummary_shim = types.ModuleType('torchsummary')
torchsummary_shim.summary = lambda *args, **kwargs: None
sys.modules['torchsummary'] = torchsummary_shim
descod_backbone = load_module('embedded_descod_backbone', SRC_ROOT / 'descod/backbone.py')
descod_diffusion = load_module('embedded_descod_diffusion', SRC_ROOT / 'descod/diffusion.py')
print('Comparative source hashes verified.')


In [ ]:
# DeScoD: cùng chiều dài 3600, không DCT hay chuẩn hóa theo source complete_eval.py.
descod_config = {'diffusion': {'beta_start': 0.0001, 'beta_end': 0.5, 'num_steps': 50, 'schedule': 'quad'}}
descod_model = descod_diffusion.DDPM(descod_backbone.ConditionalModel(80), descod_config, DEVICE).to(DEVICE)
descod_state = torch.load(WEIGHT_ROOT / 'DeScoD/model.pth', map_location='cpu', weights_only=True)
descod_model.load_state_dict(descod_state, strict=True)
descod_model.eval()
del descod_state
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
descod_pred = np.empty_like(clean)
descod_latency = np.empty(len(clean), dtype=np.float64)
with torch.inference_mode():
    for start in tqdm(range(0, len(clean), 4), desc='DeScoD DDPM50'):
        stop = min(start + 4, len(clean))
        x = torch.from_numpy(noisy[start:stop]).to(DEVICE)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        y = descod_model.denoising(x)
        torch.cuda.synchronize(); elapsed = time.perf_counter() - t0
        descod_pred[start:stop] = y.cpu().numpy()
        descod_latency[start:stop] = elapsed * 1000 / (stop-start)
assert np.isfinite(descod_pred).all()
np.savez_compressed(OUT / 'descod_predictions.npz', sample_id=sample_id,
                    prediction=descod_pred, latency_ms=descod_latency,
                    data_sha256=data_hash)
del descod_model
torch.cuda.empty_cache()
print('DeScoD inference complete.')


In [ ]:
# Nạp và chạy toàn bộ H5 bằng mã kiến trúc đi kèm.
# Kaggle Python GPU thường có TensorFlow; nếu image không có, bật Internet và cài tensorflow trước cell này.
try:
    import tensorflow as tf
    import keras
except ImportError as exc:
    raise RuntimeError('Cần TensorFlow/Keras trong Kaggle image để chạy các checkpoint .h5') from exc
print('TensorFlow', tf.__version__, 'Keras', keras.__version__)
keras.backend.clear_session()
deepfilter_models = load_module('embedded_deepfilter_models', SRC_ROOT / 'deepfilter/dl_models.py')
tcdae_models = load_module('embedded_tcdae_models', SRC_ROOT / 'tcdae/dl_models.py')
cbam_models = load_module('embedded_cbam_models', SRC_ROOT / 'cbam/dl_models.py')

KERAS_SPECS = [
    ('DeepFilter_DRNN', deepfilter_models.DRRN_denoising, 'DeepFilter-master/DRNN_.best.weights.h5', 'BW source'),
    ('DeepFilter_FCN_DAE', deepfilter_models.FCN_DAE, 'DeepFilter-master/FCN_DAE_.best.weights.h5', 'BW source'),
    ('DeepFilter_Vanilla_L', deepfilter_models.deep_filter_vanilla_linear, 'DeepFilter-master/Vanilla_L_.best.weights.h5', 'BW source'),
    ('DeepFilter_Vanilla_NL', deepfilter_models.deep_filter_vanilla_Nlinear, 'DeepFilter-master/Vanilla_NL_.best.weights.h5', 'BW source'),
    ('DeepFilter_Multibranch_LANL', deepfilter_models.deep_filter_I_LANL, 'DeepFilter-master/Multibranch_LANL_.best.weights.h5', 'BW source'),
    ('DeepFilter_Multibranch_LANLD', deepfilter_models.deep_filter_model_I_LANL_dilated, 'DeepFilter-master/Multibranch_LANLD_.best.weights.h5', 'BW source'),
    ('TCDAE', tcdae_models.Transformer_DAE, 'TCDAE/Transformer_DAE_.best.weights.h5', 'TCDAE source'),
    ('CBAM_Attention_Skip_DAE', cbam_models.AttentionSkipDAE2, 'CBAM-DAE/Attention_Skip_DAE_.best.weights.h5', 'CBAM main'),
    ('CBAM_ECA_Skip_DAE', cbam_models.ECASkipDAE, 'CBAM-DAE/ECA_Skip_DAE_.best.weights.h5', 'CBAM main'),
    ('CBAM_Transformer_DAE', tcdae_models.Transformer_DAE, 'CBAM-DAE/Transformer_DAE_.best.weights.h5', 'uncertain provenance'),
]
keras_predictions = {}
keras_latencies = {}
checkpoint_status = []
PAPER_INPUT_LENGTH = {
    'DeepFilter_FCN_DAE': 3584,
    'CBAM_Attention_Skip_DAE': 3584,
    'CBAM_ECA_Skip_DAE': 3584,
}
for name, constructor, rel, provenance in KERAS_SPECS:
    model = None
    try:
        keras.backend.clear_session()
        tf.random.set_seed(SEED)
        input_length = PAPER_INPUT_LENGTH.get(name, 3600)
        model = constructor(signal_size=input_length)
        # Subclassed CBAM models must create all variables before load_weights.
        if name.startswith('CBAM_') and 'Transformer' not in name:
            model.build((None, input_length, 1))
            _ = model(tf.zeros((1, input_length, 1)), training=False)
        model.load_weights(str(WEIGHT_ROOT / rel))
        x = np.ascontiguousarray(noisy[:, 0, :input_length, None], dtype=np.float32)
        prediction = np.empty((len(clean), x.shape[1]), dtype=np.float32)
        latency = np.empty(len(clean), dtype=np.float64)
        batch_size = 16 if name == 'DeepFilter_DRNN' else 32
        for start in tqdm(range(0, len(clean), batch_size), desc=name):
            stop = min(start+batch_size, len(clean))
            t0 = time.perf_counter()
            y = model(x[start:stop], training=False).numpy()
            elapsed = time.perf_counter() - t0
            assert y.shape == (stop-start, x.shape[1], 1), (name, y.shape)
            prediction[start:stop] = y[:, :, 0]
            latency[start:stop] = elapsed * 1000 / (stop-start)
        assert np.isfinite(prediction).all(), name
        keras_predictions[name] = prediction
        keras_latencies[name] = latency
        np.savez_compressed(OUT / (name + '_predictions.npz'), sample_id=sample_id,
                            prediction=prediction, latency_ms=latency,
                            data_sha256=data_hash)
        checkpoint_status.append(dict(model=name, file=rel, sha256=WEIGHT_HASHES[rel],
                                      provenance=provenance, input_length=input_length,
                                      status='loaded_and_evaluated', error=''))
    except Exception as exc:
        checkpoint_status.append(dict(model=name, file=rel, sha256=WEIGHT_HASHES[rel],
                                      provenance=provenance, input_length=input_length,
                                      status='FAILED', error=repr(exc)))
        print(name, 'FAILED:', repr(exc))
    finally:
        del model
        keras.backend.clear_session()
        gc.collect()
status_df = pd.DataFrame(checkpoint_status)
status_df.to_csv(OUT / 'checkpoint_status.csv', index=False)
display(status_df[['model','status','provenance','error']])


In [ ]:
# Paper protocol: 3600 points, except FCN-DAE and CBAM-DAE at 3584.
# Dùng chung METRIC_BATCH_SIZE=50 cho PRD_repo có phụ thuộc trung bình batch.
def evaluate_paper_protocol(name, prediction, latency):
    metric_length = PAPER_INPUT_LENGTH.get(name, 3600)
    pred = np.asarray(prediction)
    if pred.ndim == 3:
        pred = pred[:, 0, :]
    pred = pred[:, :metric_length]
    assert pred.shape == (len(clean), metric_length), (name, pred.shape)
    rows = []
    for start in range(0, len(clean), METRIC_BATCH_SIZE):
        stop = min(start+METRIC_BATCH_SIZE, len(clean))
        y = clean[start:stop, 0, :metric_length].astype(np.float64)
        n = noisy[start:stop, 0, :metric_length].astype(np.float64)
        p = pred[start:stop].astype(np.float64)
        ssd = np.sum((p-y)**2, axis=1)
        mad = np.max(np.abs(p-y), axis=1)
        prd_repo = 100*np.sqrt(ssd / np.sum((p-y.mean())**2, axis=1))
        prd_standard = 100*np.sqrt(ssd / np.sum((y-y.mean(axis=1, keepdims=True))**2, axis=1))
        cosine = np.sum(y*p, axis=1)/(np.linalg.norm(y,axis=1)*np.linalg.norm(p,axis=1))
        power = np.sum(y*y, axis=1)
        snr_in = 10*np.log10(power/np.sum((n-y)**2,axis=1))
        snr_out = 10*np.log10(power/ssd)
        for j in range(stop-start):
            rows.append(dict(model=name, metric_length=metric_length,
                             sample_id=int(sample_id[start+j]), record_id=record_id[start+j],
                             q=float(q_test[start+j]), SSD=float(ssd[j]), MAD=float(mad[j]),
                             PRD_repo=float(prd_repo[j]), PRD_standard=float(prd_standard[j]),
                             COS_SIM=float(cosine[j]), SNR_in=float(snr_in[j]),
                             SNR_out=float(snr_out[j]), ImSNR=float(snr_out[j]-snr_in[j]),
                             latency_ms=float(latency[start+j])))
    return pd.DataFrame(rows)

all_predictions = {'Proposed': (proposed_pred, proposed_latency),
                   'TFCDiff': (tfcdiff_pred, tfcdiff_latency),
                   'DeScoD': (descod_pred, descod_latency)}
all_predictions.update({name:(keras_predictions[name],keras_latencies[name]) for name in keras_predictions})
paper_detail = pd.concat([evaluate_paper_protocol(name,pred,lat)
                          for name,(pred,lat) in all_predictions.items()], ignore_index=True)
paper_summary = paper_detail.groupby(['model','metric_length'], sort=False)[metric_cols].mean().reset_index()
paper_summary.to_csv(OUT / 'summary_metrics_paper_protocol.csv', index=False)
assert paper_detail.groupby('model').size().eq(len(clean)).all()
assert paper_summary.set_index('model').metric_length.to_dict() == {
    name: PAPER_INPUT_LENGTH.get(name, 3600) for name in all_predictions}
display(paper_summary)
print('Successful models:', len(paper_summary), '| H5 failures:', (status_df.status == 'FAILED').sum())


In [ ]:
metadata['comparative_bundle'] = str(comparison_bundle_path)
metadata['comparative_bundle_sha256'] = comparison_bundle_hash
metadata['comparative_weight_sha256'] = WEIGHT_HASHES
metadata['comparative_source_sha256'] = COMPARATIVE_SOURCE_HASHES
metadata['paper_metric_lengths'] = paper_summary.set_index('model').metric_length.to_dict()
metadata['keras_input_preprocessing'] = 'raw noisy first 3600, except FCN-DAE and CBAM-DAE first 3584; no /200 simEMG scaling'
metadata['completed_models'] = paper_summary.model.tolist()
metadata['failed_checkpoints'] = status_df.loc[status_df.status == 'FAILED','model'].tolist()
(OUT / 'evaluation_metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print('Download summary_metrics_paper_protocol.csv and checkpoint_status.csv from', OUT)
